# GAT-LSTM and LSTM autoencoders for CHB-MIT seizure prediction

```
CHB-MIT seizure prediction with GAT-LSTM and LSTM autoencoders on SWT features, in one file (no chbmit_pipeline.py).

Everything happens here: reading the BIDS copy of CHB-MIT, labelling, filtering, artifact
rejection, feature extraction (cached), the autoencoder, baselines and the evaluation.

Data and labels
  - reads <root>/sub-<patient>/sub-<patient>_scans.tsv (file start times), the EDF files and
    the *_events.tsv seizure annotations. Seizure times are read even for recordings whose
    EDF is missing or skipped, so the 4 h exclusion around every seizure is always applied.
  - 18 bipolar channels common to all recordings (FP1-F7 ... CZ-PZ; the duplicated T8-P8
    keeps its first copy). Recordings with a referential montage (chb12 runs 27-29) get their
    bipolar channels rebuilt as A - B (the common reference cancels). A recording still missing
    any of the 18 is skipped and logged (see the channel audit).
  - FIR band-pass 0.5-45 Hz on each whole recording, then 5 s non-overlapping windows.
  - a window is dropped if any channel is above 1000 uV, any channel is flat (std < 0.5 uV),
    or it contains NaN.
  - leading seizure: starts at least 4 h after the previous seizure ended (clustered seizures
    are left out, their preictal EEG would overlap the previous seizure's aftermath).
  - preictal: window fully inside [onset - 35 min, onset - 5 min) of a leading seizure
    (30 min seizure occurrence period, 5 min prediction horizon).
  - interictal: window at least 4 h before or after every seizure (leading or not).
  - every other window is excluded.

Features per 5 s window: stationary wavelet transform (from old_work_corrige_v4)
  SWT db4, 7 levels, on each 5 s window of each of the 18 channels (1280 samples, a multiple
  of 2^7, so no padding). Bands at 256 Hz: cA7 0-1, cD7 1-2, cD6 2-4, cD5 4-8, cD4 8-16,
  cD3 16-32, cD2 32-64 Hz (cD1 64-128 Hz is left out, the 45 Hz filter empties it).
  The SWT is not decimated, so every band keeps 1280 time-aligned coefficients. As in v4, the
  RMS of each band is taken over equal blocks of time and log1p is applied: swt_steps blocks per
  window (default 5, one per second) -> 18 channels x 7 bands x swt_steps = 630 features.
  GAT: each channel is a node with 7 x swt_steps features. swt_steps=1 gives 7 per node (126).
  Cached in <feature_cache_dir>/<patient>_swt<steps>_features.npz.

Additions (settings cell)
  - supervised models logreg / gbm on the same features, trained on preictal and interictal windows
  - posclock (position + clock, no EEG) and logreg_posclock (EEG + position + clock): the
    "does EEG add over position and time" test, with its own results cell
  - baseline_min: causal per-recording baseline (features minus the mean of the previous 30 min)
  - corr_features: channel correlations as extra node features for the GAT
  - leading_gap_h 1 h (was 4 h), smooth_min 5 min (was 1 min), optional k-of-n alarm rule
  - per-patient mode by default; results go to <out_base>_pp_sop<N> or <out_base>_pooled_sop<N>

Model
  Consecutive windows are grouped into sequences (default 12 x 5 s = 1 min). A sequence
  never crosses a recording gap, a rejected window or a label change, and its score belongs
  to its LAST window, so a score only uses the past. The autoencoder is trained on
  interictal sequences only; reconstruction error is the preictal score.
  Training interictal sequences are split into fit (75%), early stopping (10%) and reference
  (15%) sets. With --split interleaved (default) this is done in 15 min blocks in rotation,
  so the reference covers every part of the day; with --split contiguous it is earliest /
  next / latest in time. An error becomes a 0-1 score: the share of reference sequences
  with a lower error.

Baselines (--methods), on the same features, folds, scaling and scored windows:
  pca          reconstruction error with --baseline-pca-dim components, last window only
  mahalanobis  distance under a Ledoit-Wolf covariance, last window only
  iforest      Isolation Forest anomaly score, last window only
  timeofday    logistic regression on the clock time alone (sin/cos of the hour), trained on
               preictal and interictal training windows. It never sees the EEG: if it scores
               close to the autoencoder, the model is mostly telling day from night.
  position     logistic regression on where the window sits in the session alone (hours since
               the patient's first window, minutes since the start of the current continuous
               stretch of recording). The control from old_work_corrige_v4: there it reached AUC
               0.70, as high as the EEG. If it scores close to the autoencoder, the labels can be
               told apart by position in the recording, not by the EEG.

Modes
  --mode pooled (default): ONE model trained on all patients together.
    - each patient's features are normalised with that patient's own development interictal
      data (median / IQR), so amplitude differences between patients do not dominate
    - holdout, never used for training or for choosing the threshold: every patient's hidden
      set (below), plus whole patients listed in --holdout-patients (all their seizures are
      scored; the earliest --calib-frac of their interictal time is used only to normalise
      their features and is not scored)
    - threshold: patient-grouped cross-validation on the development data; the patients with
      development seizures are dealt into --cv-folds groups, each group is scored by a model
      trained on the other patients, and the threshold is chosen on all those scores
    - one final model on all development data, scored once on the holdout
    - outputs: pooled_<method>_{cv,holdout}_{seizures,patients}.csv, pooled_overall.json,
      pooled_lstm_ae.keras
  --mode per-patient: one model per patient, evaluated as below (lstm_ae_summary.csv).

Evaluation (per-patient mode; pooled mode uses the same hidden sets, alarms and tests)
  - hidden set (patients with >= 3 usable seizures): the last usable seizure + the latest
    20% of interictal time; the hour of interictal data just before it is dropped. Never
    used for training or tuning.
  - leave-one-seizure-out on the remaining (development) data: each fold tests one seizure's
    preictal period plus a share of the interictal time. --split interleaved (default): 1 h
    blocks (--block-min) dealt to the folds in rotation, so every fold trains on day AND night
    EEG; training interictal windows within 10 min (--gap-min) of a test window are dropped.
    --split contiguous: one consecutive block of interictal time per fold (the old behaviour).
    With contiguous blocks a fold can train on one part of the day and test on another, and
    the scores then measure vigilance state (awake / asleep) rather than preictal change.
  - alarms: causal 1 min moving average of the score; an alarm when it reaches the threshold,
    then 30 min refractory. A seizure is predicted if an alarm falls in its preictal period;
    other alarms are false alarms, counted per interictal hour.
  - threshold, chosen in cross-validation: highest sensitivity with at most --max-fpr false
    alarms per hour (ties: fewer false alarms, then the higher threshold).
  - one final model trained on all development data, scored once on the hidden set.
  - random predictor (Schelter et al. 2006): Poisson alarms at the same false-alarm rate hit
    a 30 min SOP with probability 1 - exp(-FPR * SOP). Per-patient p-value = P(random
    predicts >= the seizures we predicted); the pooled p-value treats every seizure as an
    independent trial with its patient's chance. A patient with FPR = 0 makes any predicted
    seizure significant, so read those with their interictal hours.
  - summaries pooled (all seizures / all hours) and as per-patient means.

Usage
    pip install mne numpy pandas scipy scikit-learn PyWavelets tensorflow
    python lstm_autoencoder.py --root "C:\\dataset chb\\.nemar" --out results_ae
    python lstm_autoencoder.py --root "C:\\dataset chb\\.nemar" --out results_ae --holdout-patients chb05 chb10 chb20
    python lstm_autoencoder.py --root "C:\\dataset chb\\.nemar" --out results_ae --mode per-patient --patients chb01
    python lstm_autoencoder.py --root "C:\\dataset chb\\.nemar" --out results_ae --features-only
    python lstm_autoencoder.py --root "C:\\dataset chb\\.nemar" --out results_ae --methods lstm timeofday
    python lstm_autoencoder.py --check-gpu
```

## Settings
The default output folder is `results_gat`. Features are the stationary wavelet transform of old_work_corrige_v4 (db4, 7 levels), cached in `feature_cache_dir` (default `results_gat\features_swt`, separate from the old 829-feature cache, which is no longer used). Each preictal length (SOP) in `sop_list` is run in turn, with its results in `<out_base>_pp_sop<N>` (per-patient) or `<out_base>_pooled_sop<N>` (pooled) and its feature cache in `<cache_base>_sop<N>` (the cache keeps only labelled windows, so every SOP needs its own). Set `methods=["gat"]` to run only GAT, or `methods=["gat", "lstm"]` for a direct comparison. Then use **Run All**.

In [ ]:
# ---- settings: edit these, then run all cells -------------------------------------------
from types import SimpleNamespace

args = SimpleNamespace(
    root=r"C:\dataset chb\.nemar",   # folder that contains sub-chb01 ... sub-chb24
    cache_base=r"results_gat2\features",     # feature cache, one folder per SOP: <cache_base>_sop10, ...
    out_base="results_gat2",         # results: <out_base>_pp_sop10 (per-patient), <out_base>_pooled_sop10 (pooled)
    sop_list=[10, 20, 30],            # preictal lengths (SOP, minutes) to run, one after the other
                                      # preictal = [onset - SOP - 5 min, onset - 5 min); refractory = SOP
    patients=None,                    # None = all patients found; or e.g. ["chb01", "chb05"]

    mode="both",                      # "both": per-patient, then pooled, in one run (no kernel restart)
                                      # "per-patient": one model per patient; "pooled": one model for all
    holdout_patients=[],              # pooled: whole patients never used for training, e.g. ["chb05", "chb10"]
    cv_folds=4,                       # pooled: patient groups used to choose the threshold
    calib_frac=0.2,                   # pooled: share of a held-out patient's interictal time used only to normalise
    methods=["logreg", "gbm", "gat", "lstm", "posclock", "logreg_posclock", "position", "timeofday"],
                                      # supervised (trained on preictal AND interictal): logreg, gbm
                                      # unsupervised autoencoders (interictal only): gat, lstm
                                      # controls without EEG: posclock (position + clock), position, timeofday
                                      # logreg_posclock: EEG + position + clock -> does EEG add over posclock?
                                      # also available: "pca", "mahalanobis", "iforest"
    features_only=False,              # True: only extract and cache features
    save_models=True,                 # save final LSTM models to <out>/models for predicting new recordings
    use_gpu=True,                     # train on the GPU when TensorFlow sees one (see the GPU setup cell)

    leading_gap_h=1.0,                # a seizure is used if it starts >= this long after the previous one ended
                                      # (was 4.0: more seizures now)
    interictal_gap_h=4.0,             # a window is interictal only if it is >= this many hours from every seizure
                                      # (4 h is the usual choice; smaller = more interictal, but closer to seizures)
    baseline_min=30,                  # causal per-recording baseline: subtract the mean features of the previous
                                      # 30 min of the same recording (0 = off). Removes slow drifts and offsets
                                      # between files; windows with < 5 min of past in their recording are dropped
    corr_features=True,               # add each channel's correlation with the 18 channels (18 per GAT node)
    smooth_min=5,                     # moving average of the score before the alarm threshold (was 1 min)
    vote_k=0, vote_n=0,               # k-of-n alarm rule instead of the average, e.g. vote_k=8, vote_n=10
                                      # (alarm when 8 of the last 10 scores are above the threshold; 0 = off)
    logreg_C=0.01,                    # L2 strength of the logistic regressions (smaller = stronger)
    max_train_windows=100000,         # cap on windows for the supervised models (all preictal kept up to half)
    swt_steps=5,                      # SWT (db4, 7 levels) RMS blocks per 5 s window: 5 = one per second
                                      # -> 18 x 7 x 5 = 630 features (35 per GAT node); 1 -> 126 (7 per node)
    scaler="standard",                # "standard" or "robust"
    seq_len=12,                       # windows per sequence (12 x 5 s = 1 min)
    stride=3,                         # step between training sequences
    units=64,                         # LSTM units
    latent=32,                        # bottleneck size
    he_leaky=False,                   # True: He init + LeakyReLU(0.1) on the input Dense layer (ablation; LSTMs keep tanh)
    epochs=50,                        # maximum; early stopping on the validation set
    max_train_seq=20000,              # cap on training sequences (lower it if memory runs out)
    gat_max_train_seq=5000,           # lower cap because graph tensors are larger
    baseline_pca_dim=10,              # components for the PCA baseline
    seed=0,

    split="interleaved",              # "interleaved" or "contiguous" interictal CV blocks
    block_min=60,                     # interleaved block length (minutes)
    gap_min=10,                       # drop training interictal windows this close to test ones
    max_fpr=0.2,                      # false alarms per hour allowed when choosing the threshold
    holdout_frac=0.2,                 # share of interictal time in each patient's hidden set
    check_labels=True,                # assert the labelling rules (horizon, margins) on every recording
)

## Imports
`pip install mne numpy pandas scipy scikit-learn PyWavelets tensorflow`

In [ ]:
import json

import joblib
import os
from dataclasses import asdict, dataclass

import mne
import numpy as np
import pandas as pd
import pywt
from scipy import signal, stats
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import RobustScaler, StandardScaler

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

mne.set_log_level("ERROR")

## GPU setup
Keras trains the LSTM autoencoder on an NVIDIA GPU automatically when TensorFlow sees one. This cell reports what is available, turns on memory growth, and explains how to enable the GPU if it is missing. `use_gpu=False` in the settings forces the CPU. Feature extraction always runs on the CPU (it is cached after the first run).

In [ ]:
import platform
import time


def setup_gpu(use_gpu=True, benchmark=False):
    """Choose the device for TensorFlow. Keras trains on the GPU automatically when one is visible;
    this cell runs before any model is built because TensorFlow only accepts memory settings before
    the GPU is first used. Returns "GPU" or "CPU"."""
    gpus = tf.config.list_physical_devices("GPU")
    print(f"TensorFlow {tf.__version__}, built with CUDA: {tf.test.is_built_with_cuda()}")
    if not use_gpu:
        try:
            tf.config.set_visible_devices([], "GPU")
        except RuntimeError:
            print("the GPU is already in use: restart the kernel to switch to CPU")
        print("use_gpu=False: training on CPU")
        return "CPU"
    if not gpus:
        print("no GPU visible to TensorFlow: training on CPU")
        if platform.system() == "Windows":
            print("Native Windows: TensorFlow 2.11 and later have no GPU support. Two ways to use an NVIDIA GPU:\n"
                  "  1. WSL2 (Ubuntu) with the NVIDIA driver for Windows, then inside WSL:\n"
                  "     pip install \"tensorflow[and-cuda]\"   (run this notebook from WSL, e.g. with VS Code)\n"
                  "  2. a Python 3.10 environment with tensorflow==2.10, CUDA 11.2 and cuDNN 8.1")
        else:
            print('install the GPU build: pip install "tensorflow[and-cuda]" (needs the NVIDIA driver; check with nvidia-smi)')
        return "CPU"
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)   # take GPU memory as needed, not all at once
        except RuntimeError:
            pass
        d = tf.config.experimental.get_device_details(gpu)
        print(f"GPU: {d.get('device_name', gpu.name)} (compute capability {d.get('compute_capability', '?')}), "
              f"memory growth on")
    print("the LSTM autoencoder will train on the GPU (fast cuDNN LSTM kernels: default activations, no recurrent dropout)")
    if benchmark:
        a = tf.random.normal((4000, 4000))
        for dev in ("/CPU:0", "/GPU:0"):
            with tf.device(dev):
                tf.matmul(a, a).numpy()
                t0 = time.perf_counter()
                for _ in range(10):
                    r = tf.matmul(a, a)
                r.numpy()
                print(f"  10 x 4000x4000 matmul on {dev}: {time.perf_counter() - t0:.2f} s")
    return "GPU"


DEVICE = setup_gpu(args.use_gpu, benchmark=False)   # benchmark=True also times CPU vs GPU

## Constants and configuration

In [ ]:
MIN_DEV_INTERICTAL_H = 1.0          # no hidden set if it would leave less development interictal data
MIN_TRAIN_PER_CLASS = 20             # a fold needs this many training windows of each class
FS = 256
CHANNELS = ["FP1-F7", "F7-T7", "T7-P7", "P7-O1", "FP1-F3", "F3-C3", "C3-P3", "P3-O1", "FP2-F4", "F4-C4",
            "C4-P4", "P4-O2", "FP2-F8", "F8-T8", "T8-P8", "P8-O2", "FZ-CZ", "CZ-PZ"]
SWT_WAVELET, SWT_LEVEL = "db4", 7      # as in old_work_corrige_v4
SWT_BANDS = ["cA7", "cD7", "cD6", "cD5", "cD4", "cD3", "cD2"]   # 0-1, 1-2, 2-4, 4-8, 8-16, 16-32, 32-64 Hz
FEATURE_VERSION = "swt-v2"
METHODS = ["logreg", "gbm", "gat", "lstm", "posclock", "logreg_posclock", "position", "timeofday",
           "pca", "mahalanobis", "iforest"]
SUPERVISED = ("logreg", "gbm", "posclock", "logreg_posclock")
LABEL_KEYS = ("win_s", "l_freq", "h_freq", "max_abs_uv", "min_std_uv", "preictal_min", "sph_min",
              "interictal_gap_h", "leading_gap_h")     # the settings a feature cache depends on
BASELINE_MIN_PAST_MIN = 5.0                 # a window needs this much past in its recording for the baseline
FIT_FRAC, VAL_FRAC = 0.75, 0.10             # the rest (15%) is the reference
INNER_BLOCK_MIN = 15                        # interleaved fit / early-stop / reference blocks


@dataclass
class Config:
    win_s: float = 5.0
    l_freq: float = 0.5
    h_freq: float = 45.0
    max_abs_uv: float = 1000.0
    min_std_uv: float = 0.5
    preictal_min: float = 35.0              # preictal starts this long before onset ...
    sph_min: float = 5.0                    # ... and ends this long before onset
    interictal_gap_h: float = 4.0           # interictal: at least this far from any seizure
    leading_gap_h: float = 4.0              # leading seizure: this long after the previous one ended
    smooth_windows: int = 12                # 12 x 5 s = 1 min moving average
    vote_k: int = 0                         # k-of-n alarm rule (0 = off, use the moving average)
    vote_n: int = 0
    refractory_min: float = 30.0
    max_fpr: float = 0.2
    holdout_frac: float = 0.2
    holdout_gap_h: float = 1.0

    @property
    def sop_min(self):
        return self.preictal_min - self.sph_min

    @property
    def win_ns(self):
        return int(self.win_s * 1e9)


def make_config(args):
    """Config for the current SOP (args.sop_min): preictal from onset - (SOP + SPH) to onset - SPH,
    refractory period after an alarm = SOP; leading-seizure gap, smoothing and alarm rule from args."""
    sph = Config.sph_min
    return Config(max_fpr=args.max_fpr, holdout_frac=args.holdout_frac,
                  preictal_min=args.sop_min + sph, refractory_min=float(args.sop_min),
                  leading_gap_h=float(getattr(args, "leading_gap_h", 4.0)),
                  interictal_gap_h=float(getattr(args, "interictal_gap_h", 4.0)),
                  smooth_windows=max(1, int(round(getattr(args, "smooth_min", 1) * 60 / Config.win_s))),
                  vote_k=int(getattr(args, "vote_k", 0)), vote_n=int(getattr(args, "vote_n", 0)))


def mode_tag(args):
    return "pooled" if args.mode == "pooled" else "pp"


def use_sop(args, sop):
    """Point args at one SOP: its settings, results folder (per mode) and feature cache folder.
    The features do not depend on the mode, so both modes share one cache per SOP."""
    args.sop_min = sop
    args.out = f"{args.out_base}_{mode_tag(args)}_sop{sop}"
    args.feature_cache_dir = f"{args.cache_base}_sop{sop}"
    return args


# mode="both" runs per-patient, then pooled. args.mode always holds ONE mode (the one being run or
# shown); the experiment, results and figure cells below loop over MODES. Run all cells after
# changing mode in the settings.
MODES = ["per-patient", "pooled"] if args.mode == "both" else [args.mode]
args.mode = MODES[0]

use_sop(args, args.sop_list[0])     # the sanity-check cells below use the first SOP


## Reading CHB-MIT (BIDS)

In [ ]:
import re


def channel_plan(names):
    """How to get each of the 18 bipolar channels from a recording's raw channel names.
    {canonical: (raw_name,)}         the bipolar channel is in the file (exact name first, then the
                                     MNE duplicate names T8-P8-0 / T8-P8-1: first copy wins)
    {canonical: (raw_a, raw_b)}      rebuilt as A - B from a referential montage (chb12 runs 27-29:
                                     'FP1-CS2', 'F7-CS2' ... or 'FP1', 'F7' ...; the common
                                     reference cancels in the subtraction)
    Channels that cannot be found or rebuilt are left out."""
    plan = {}
    for n in names:                                         # exact bipolar name
        u = n.strip().upper()
        if u in CHANNELS and u not in plan:
            plan[u] = (n,)
    for n in names:                                         # MNE duplicate suffix -0, -1, ...
        u = re.sub(r"-\d+$", "", n.strip().upper())
        if u in CHANNELS and u not in plan:
            plan[u] = (n,)
    elec = {}                                               # single electrodes (referential montage)
    for n in names:
        u = re.sub(r"-CS2$", "", n.strip().upper())
        if u == "01":                                       # zero typed instead of the letter O
            u = "O1"
        if re.fullmatch(r"[A-Z]+\d*Z?\d*", u) and u not in elec:
            elec[u] = n
    for c in CHANNELS:
        if c not in plan:
            a, b = c.split("-")
            if a in elec and b in elec:
                plan[c] = (elec[a], elec[b])
    return plan


def channel_map(names):
    """Raw channel name -> canonical name for the channels present as bipolar pairs
    (kept for compatibility; channel_plan also rebuilds referential montages)."""
    return {v[0]: c for c, v in channel_plan(names).items() if len(v) == 1}


def load_bipolar(path, plan, cfg):
    """The 18 bipolar channels of one recording in uV, CHANNELS order: native channels as they are,
    referential montages rebuilt as A - B. Resampled to FS and band-passed like every recording
    (filtering is linear, so filtering before the subtraction is the same as after)."""
    raw = mne.io.read_raw_edf(path, preload=False)
    needed = sorted({n for v in plan.values() for n in v}, key=raw.ch_names.index)
    raw.pick(needed).load_data()
    if raw.info["sfreq"] != FS:
        raw.resample(FS)
    raw.filter(cfg.l_freq, cfg.h_freq, fir_design="firwin")
    d = raw.get_data(units="uV")
    row = {n: i for i, n in enumerate(raw.ch_names)}
    return np.stack([d[row[plan[c][0]]] if len(plan[c]) == 1 else d[row[plan[c][0]]] - d[row[plan[c][1]]]
                     for c in CHANNELS])


def to_naive(ts):
    ts = pd.Timestamp(ts)
    ts = ts.tz_convert(None) if ts.tzinfo is not None else ts
    return ts.as_unit("ns")


def build_catalog(root, patient):
    """One entry per recording in scans.tsv: start, end, EDF path, usable channels, seizures."""
    sub = os.path.join(root, f"sub-{patient}")
    scans = pd.read_csv(os.path.join(sub, f"sub-{patient}_scans.tsv"), sep="\t", encoding="utf-8-sig")
    catalog = []
    for _, r in scans.iterrows():
        path = os.path.join(sub, *str(r["filename"]).split("/"))
        start = pd.to_datetime(r["acq_time"], errors="coerce")
        if pd.isna(start):
            print(f"  {os.path.basename(path)}: no start time in scans.tsv, skipped")
            continue
        start = to_naive(start)
        rec = dict(path=path, name=os.path.basename(path), start=start, end=None, has_edf=False,
                   channels_ok=False, plan={}, derived=0, seizures=[])
        ev_path = path.replace("_eeg.edf", "_events.tsv")
        if os.path.exists(ev_path):
            ev = pd.read_csv(ev_path, sep="\t", encoding="utf-8-sig")
            if "trial_type" in ev:
                ev = ev[ev["trial_type"].astype(str).str.contains("seizure", case=False)]
            for _, e in ev.iterrows():
                on = start + pd.to_timedelta(float(e["onset"]), unit="s")
                rec["seizures"].append((on, on + pd.to_timedelta(float(e["duration"]), unit="s")))
        if os.path.exists(path):
            raw = mne.io.read_raw_edf(path, preload=False)
            rec["end"] = start + pd.to_timedelta(raw.n_times / raw.info["sfreq"], unit="s")
            rec["has_edf"] = True
            rec["plan"] = channel_plan(raw.ch_names)
            rec["derived"] = sum(len(v) == 2 for v in rec["plan"].values())
            rec["channels_ok"] = len(rec["plan"]) == len(CHANNELS)
        catalog.append(rec)
    return sorted(catalog, key=lambda r: r["start"])


def seizure_events(catalog, cfg):
    """All seizures in time order, with id (1, 2, ...) and the leading flag."""
    sz = sorted(s for rec in catalog for s in rec["seizures"])
    out, prev_end = [], None
    for i, (on, end) in enumerate(sz):
        leading = prev_end is None or (on - prev_end) >= pd.Timedelta(hours=cfg.leading_gap_h)
        out.append(dict(id=i + 1, onset=on.value, end=end.value, leading=leading))
        prev_end = end if prev_end is None else max(prev_end, end)
    return out


def label_windows(ws, seizures, cfg):
    """ws: window start times (ns). Returns label (1 preictal, 0 interictal, -1 excluded) and
    seizure id (preictal only, else -1)."""
    win, gap = cfg.win_ns, int(cfg.interictal_gap_h * 3600e9)
    lab, sid = np.full(len(ws), -1), np.full(len(ws), -1)
    far = np.ones(len(ws), bool)
    for s in seizures:
        far &= (ws + win <= s["onset"] - gap) | (ws >= s["end"] + gap)
    lab[far] = 0
    for s in seizures:
        if s["leading"]:
            pre = (ws >= s["onset"] - int(cfg.preictal_min * 60e9)) & (ws + win <= s["onset"] - int(cfg.sph_min * 60e9))
            lab[pre], sid[pre] = 1, s["id"]
    return lab, sid


def check_labels(ws, lab, sid, seizures, cfg):
    """Assert the labelling rules on one recording's windows (the old notebook's horizon-leak check,
    extended to the interictal margin). Raises AssertionError with the first offending window."""
    win = cfg.win_ns
    by_id = {s["id"]: s for s in seizures}
    gap = int(cfg.interictal_gap_h * 3600e9)
    for i in np.flatnonzero(lab == 1):
        s = by_id[sid[i]]
        assert s["leading"], f"preictal window {i} belongs to a non-leading seizure {s['id']}"
        assert ws[i] + win <= s["onset"] - int(cfg.sph_min * 60e9), \
            f"preictal window {i} ends inside the {cfg.sph_min:g} min horizon before seizure {s['id']}"
        assert ws[i] >= s["onset"] - int(cfg.preictal_min * 60e9), \
            f"preictal window {i} starts before the preictal period of seizure {s['id']}"
        for o in seizures:
            assert ws[i] + win <= o["onset"] or ws[i] >= o["end"], f"preictal window {i} overlaps seizure {o['id']}"
    for i in np.flatnonzero(lab == 0):
        for o in seizures:
            assert ws[i] + win <= o["onset"] - gap or ws[i] >= o["end"] + gap, \
                f"interictal window {i} is within {cfg.interictal_gap_h:g} h of seizure {o['id']}"


## Features: stationary wavelet transform (from old_work_corrige_v4)
Same transform as v4 cell 11: SWT db4 level 7 per channel, bands cA7 and cD7 to cD2, RMS over equal time blocks, then log1p. v4 used 10 s windows and 256 blocks for a 2-D CNN; here a window is 5 s and the autoencoder needs a compact vector, so `swt_steps` blocks per window (default 5, one per second).

Two additions:
- `corr_features=True`: each channel's correlation with the 18 channels (18 more values per channel, the GAT's cross-channel input).
- `baseline_min=30`: a causal per-recording baseline. Every window's features minus the mean of the previous 30 min of the same recording. Slow drifts (sleep, time of day) and offsets between files, which let the `position` control separate the classes, are removed; what remains is change relative to the recent past. It uses only the past, so it works in real time. Windows with less than 5 min of past in their recording are dropped.

In [ ]:
def feature_names(steps=None, corr=None):
    """{channel}|swt_{band}|t{k}, then {channel}|corr|{other} (with corr_features), channel first so
    the GAT can group each channel's features into its node."""
    steps = steps or args.swt_steps
    corr = getattr(args, "corr_features", False) if corr is None else corr
    names = []
    for ch in CHANNELS:
        names += [f"{ch}|swt_{b}|t{k}" for b in SWT_BANDS for k in range(steps)]
        if corr:
            names += [f"{ch}|corr|{o}" for o in CHANNELS]
    return names


def window_features(x, steps=None, corr=None):
    """x: (n_windows, 18, samples) in uV -> (n_windows, 18 * (7 * steps [+ 18])).
    SWT db4 level 7 (not decimated, so every band keeps all samples, aligned in time), keep cA7 and
    cD7..cD2, RMS over `steps` equal blocks, log1p. The same as dwt_transform_window in
    old_work_corrige_v4, with fewer time blocks. With corr: each channel's Pearson correlation with
    the 18 channels in the window (cross-channel information for the GAT nodes)."""
    steps = steps or args.swt_steps
    corr = getattr(args, "corr_features", False) if corr is None else corr
    nw, nc, n = x.shape
    assert n % (2 ** SWT_LEVEL) == 0, f"SWT: window length {n} must be a multiple of {2 ** SWT_LEVEL}"
    assert n % steps == 0, f"window length {n} is not a multiple of swt_steps={steps}"
    sw = pywt.swt(x.astype(np.float64), SWT_WAVELET, level=SWT_LEVEL, axis=-1, trim_approx=True)
    bands = np.stack(sw[:SWT_LEVEL], axis=2)                 # [cA7, cD7, ..., cD2]: (nw, 18, 7, n), cD1 dropped
    rms = np.sqrt((bands.reshape(nw, nc, SWT_LEVEL, steps, n // steps) ** 2).mean(-1))
    parts = [np.log1p(rms).reshape(nw, nc, -1)]
    if corr:
        z = (x - x.mean(-1, keepdims=True)) / (x.std(-1, keepdims=True) + 1e-12)
        parts.append(np.einsum("wit,wjt->wij", z, z) / n)    # (nw, 18, 18)
    return np.nan_to_num(np.concatenate(parts, -1)).reshape(nw, -1).astype(np.float32)


def windows_ok(data, idx, n, cfg):
    """Artifact rules for the windows idx of one recording: finite, every channel <= max_abs_uv and
    no flat channel (std < min_std_uv)."""
    ok = np.zeros(len(idx), bool)
    for a in range(0, len(idx), 256):
        segs = np.stack([data[:, i * n:(i + 1) * n] for i in idx[a:a + 256]])
        ok[a:a + 256] = (np.isfinite(segs).all((1, 2)) & (np.abs(segs).max((1, 2)) <= cfg.max_abs_uv)
                         & (segs.std(-1).min(-1) >= cfg.min_std_uv))
    return ok


def features_of(data, idx, n, steps=None, corr=None):
    """Features of the windows idx of one recording, in chunks (bounded memory)."""
    out = [window_features(np.stack([data[:, i * n:(i + 1) * n] for i in idx[a:a + 128]]), steps, corr)
           for a in range(0, len(idx), 128)]
    return np.concatenate(out) if out else np.zeros((0, len(feature_names(steps, corr))), np.float32)


def causal_baseline(X, tw, base_min, win_s=5.0, min_past_min=BASELINE_MIN_PAST_MIN):
    """X minus the mean of the same recording's valid windows in the previous base_min minutes
    (strictly before each window, so it only uses the past). tw: window start times (ns), sorted.
    Returns (detrended X, valid): valid is False where under min_past_min of past windows exist."""
    cs = np.vstack([np.zeros((1, X.shape[1])), np.cumsum(X.astype(np.float64), 0)])
    lo = np.searchsorted(tw, tw - int(base_min * 60e9), side="left")
    hi = np.arange(len(tw))
    cnt = hi - lo
    mean = (cs[hi] - cs[lo]) / np.maximum(cnt, 1)[:, None]
    valid = cnt >= int(min_past_min * 60 / win_s)
    return (X - mean).astype(np.float32), valid


def recording_features(rec, seizures, cfg):
    dur = (rec["end"] - rec["start"]).total_seconds()
    ws = pd.DatetimeIndex(rec["start"] + pd.to_timedelta(np.arange(0, dur - cfg.win_s + 1e-9, cfg.win_s), unit="s"))
    ws = ws.as_unit("ns").asi8                              # ns, like the seizure times (Timestamp.value)
    lab, sid = label_windows(ws, seizures, cfg)
    if CHECK_LABELS:
        check_labels(ws, lab, sid, seizures, cfg)
    if not (lab >= 0).any():
        return None
    data = load_bipolar(rec["path"], rec["plan"], cfg)
    n = int(cfg.win_s * FS)
    base = getattr(args, "baseline_min", 0)
    idx = np.arange(min(len(ws), data.shape[1] // n))
    if not base:                                            # only the labelled windows are needed
        idx = idx[lab[idx] >= 0]
    ok = windows_ok(data, idx, n, cfg)
    n_rej = int((~ok & (lab[idx] >= 0)).sum())
    idx = idx[ok]
    if len(idx) == 0:
        return None
    X = features_of(data, idx, n)
    if base:                                                # baseline from every valid window, labelled or not
        X, valid = causal_baseline(X, ws[idx], base, cfg.win_s)
        keep = valid & (lab[idx] >= 0)
        X, idx = X[keep], idx[keep]
        if len(idx) == 0:
            return None
    return X, lab[idx], sid[idx], ws[idx], n_rej


CHECK_LABELS = True     # set from args.check_labels in run()


def cache_is_current(info, version, cfg, steps=None):
    """A cache is reused only if it was made with the same feature version and the same labelling and
    preprocessing settings (smoothing, alarms and the like do not matter), and it did not skip
    recordings for missing channels (older caches skipped the referential chb12 runs that are now
    rebuilt, so those patients are extracted again)."""
    old, new = info.get("config") or {}, asdict(cfg)
    return (info.get("feature_version") == version and all(old.get(k) == new[k] for k in LABEL_KEYS)
            and info.get("swt_steps") == (steps or args.swt_steps)
            and bool(info.get("corr_features")) == bool(getattr(args, "corr_features", False))
            and info.get("baseline_min", 0) == getattr(args, "baseline_min", 0)
            and not info.get("skipped_missing_channels"))


def patient_features(root, patient, cfg, cache_dir):
    """Features, labels, seizure ids and window start times for one patient (cached)."""
    os.makedirs(cache_dir, exist_ok=True)              # the cache folder may not exist yet (first call)
    tag = (f"swt{args.swt_steps}" + ("_corr" if getattr(args, "corr_features", False) else "")
           + (f"_base{args.baseline_min:g}" if getattr(args, "baseline_min", 0) else ""))
    cache = os.path.join(cache_dir, f"{patient}_{tag}_features.npz")
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=True)
        info = json.loads(str(z["info"]))
        if cache_is_current(info, FEATURE_VERSION, cfg):
            return z["X"], z["y"], z["g"], z["t"], info
        print(f"  {patient}: cached features out of date (settings, version or skipped recordings), extracting again")
    catalog = build_catalog(root, patient)
    seizures = seizure_events(catalog, cfg)
    info = dict(feature_version=FEATURE_VERSION, swt_steps=args.swt_steps,
                corr_features=bool(getattr(args, "corr_features", False)),
                baseline_min=getattr(args, "baseline_min", 0), config=asdict(cfg), recordings=len(catalog),
                recording_hours=sum((r["end"] - r["start"]).total_seconds() for r in catalog if r["has_edf"]) / 3600,
                seizures=len(seizures), leading_seizures=sum(s["leading"] for s in seizures),
                missing_edf=[r["name"] for r in catalog if not r["has_edf"]],
                skipped_missing_channels=[r["name"] for r in catalog if r["has_edf"] and not r["channels_ok"]],
                rebuilt_referential=[r["name"] for r in catalog if r["channels_ok"] and r["derived"]],
                rejected_windows=0)
    Xs, ys, gs, ts = [], [], [], []
    for rec in catalog:
        if not (rec["has_edf"] and rec["channels_ok"]):
            continue
        out = recording_features(rec, seizures, cfg)
        if out is None:
            continue
        X, y, g, t, n_rej = out
        Xs.append(X); ys.append(y); gs.append(g); ts.append(t)
        info["rejected_windows"] += n_rej
        print(f"  {rec['name']}: {int((y == 1).sum())} preictal, {int((y == 0).sum())} interictal windows")
    for name in info["skipped_missing_channels"]:
        print(f"  {name}: missing some of the 18 channels, skipped")
    if Xs:
        X, y, g, t = (np.concatenate(a) for a in (Xs, ys, gs, ts))
    else:
        X, y, g, t = np.zeros((0, len(feature_names())), np.float32), np.zeros(0, int), np.zeros(0, int), np.zeros(0, np.int64)
    info["interictal_hours"] = float((y == 0).sum() * cfg.win_s / 3600)
    info["usable_seizures"] = int(len(np.unique(g[y == 1])))
    np.savez_compressed(cache, X=X, y=y, g=g, t=t, info=json.dumps(info, default=str),
                        feature_names=np.array(feature_names()))
    return X, y, g, t, info


def load_patient(args, pt, cfg, feat_dir):
    """SWT features, labels, seizure ids, times and info for one patient (cached in feat_dir)."""
    return patient_features(args.root, pt, cfg, feat_dir)

## Hidden set, smoothing, alarms

In [ ]:
def split_holdout(y, g, t, cfg):
    """(dev mask, hidden mask), or None with fewer than 3 usable seizures or when holding data
    out would leave under MIN_DEV_INTERICTAL_H of development interictal time. Hidden = the last
    usable seizure + the latest holdout_frac of interictal time; the holdout_gap_h of
    interictal data just before it is dropped."""
    sz = np.unique(g[y == 1])
    inter_t = np.sort(t[y == 0])
    if len(sz) < 3 or len(inter_t) == 0:
        return None
    cut = inter_t[int(len(inter_t) * (1 - cfg.holdout_frac))]
    hid = ((y == 1) & (g == sz[-1])) | ((y == 0) & (t >= cut))
    drop = (y == 0) & (t < cut) & (t >= cut - int(cfg.holdout_gap_h * 3600e9))
    dev = ~hid & ~drop
    if (dev & (y == 0)).sum() * cfg.win_s / 3600 < MIN_DEV_INTERICTAL_H or not (hid & (y == 0)).any():
        return None                                         # too little interictal data to hold some out
    return dev, hid


def smooth(p, t, cfg):
    """Causal score smoothing before the threshold (gaps respected).
    Default: moving average over the last smooth_windows windows of time.
    vote_k > 0: the vote_k-th highest score of the last vote_n windows, so 'smoothed >= threshold'
    means 'at least vote_k of the last vote_n scores are >= threshold' (k-of-n alarm rule)."""
    if len(p) == 0:
        return p
    o = np.argsort(t)
    ps, ts = p[o], t[o]
    out = np.empty_like(ps, dtype=float)
    if getattr(cfg, "vote_k", 0) > 0:
        k, n = cfg.vote_k, max(cfg.vote_n, cfg.vote_k)
        lo = np.searchsorted(ts, ts - n * cfg.win_ns, side="right")
        v = np.zeros(len(ps))
        for i in range(len(ps)):
            w = ps[lo[i]:i + 1]
            v[i] = np.partition(w, len(w) - k)[len(w) - k] if len(w) >= k else 0.0
        out[o] = v
        return out
    c = np.r_[0, np.cumsum(ps)]
    lo = np.searchsorted(ts, ts - cfg.smooth_windows * cfg.win_ns, side="right")
    hi = np.arange(1, len(ps) + 1)
    out[o] = (c[hi] - c[lo]) / (hi - lo)
    return out


def alarm_metrics(ps, yt, tt, thr, cfg):
    """(seizure predicted, false alarms, interictal hours) for one test set."""
    o = np.argsort(tt)
    ps, yt, tt = ps[o], yt[o], tt[o]
    refr = cfg.refractory_min * 60e9
    last, detected, fa = -np.inf, False, 0
    for i in np.flatnonzero(ps >= thr):
        if tt[i] - last >= refr:
            last = tt[i]
            if yt[i] == 1:
                detected = True
            else:
                fa += 1
    return detected, fa, float((yt == 0).sum() * cfg.win_s / 3600)


def choose_threshold(fold_preds, cfg):
    """Highest sensitivity with at most max_fpr false alarms per hour; among equally sensitive
    thresholds the lowest one (the most sensitive setting the false-alarm budget allows), so a
    model that caught nothing in CV can still raise alarms later. If no threshold is within
    max_fpr: fewest false alarms. The grid is finer near 1, where rank scores need it."""
    best, best_key = 1.0, None
    for thr in np.unique(np.r_[np.linspace(0, 1, 201), 1 - np.logspace(-4, -1.3, 60)]):
        det = fa = h = 0
        for ps, yt, tt in fold_preds:
            d, f, hh = alarm_metrics(ps, yt, tt, thr, cfg)
            det += d; fa += f; h += hh
        sens, fpr = det / len(fold_preds), fa / max(h, 1e-9)
        ok = fpr <= cfg.max_fpr
        key = (True, sens, -thr) if ok else (False, -fpr, thr)
        if best_key is None or key > best_key:
            best, best_key = float(thr), key
    return best

## Sequences

In [ ]:
def sequence_index(y, g, t, mask, seq_len, stride, win_ns, pid=None):
    """(n_seq, seq_len) matrix of window indices: runs of seq_len contiguous windows inside
    mask that share one patient, one label and one seizure id. Rows are in (patient, time)
    order; column -1 is the window the sequence scores."""
    sel = np.where(mask)[0]
    p = np.zeros(len(t), int) if pid is None else pid
    sel = sel[np.lexsort((t[sel], p[sel]))]
    if len(sel) < seq_len:
        return np.zeros((0, seq_len), int)
    breaks = np.where((np.diff(t[sel]) > 1.5 * win_ns) | (np.diff(t[sel]) < 0) | (np.diff(y[sel]) != 0)
                      | (np.diff(g[sel]) != 0) | (np.diff(p[sel]) != 0))[0] + 1
    rows = []
    for seg in np.split(np.arange(len(sel)), breaks):
        starts = np.arange(0, len(seg) - seq_len + 1, stride)
        if len(starts):
            rows.append(sel[seg[starts][:, None] + np.arange(seq_len)])
    return np.concatenate(rows) if rows else np.zeros((0, seq_len), int)

## Models

The notebook supports the original flat-feature LSTM autoencoder (`method="lstm"`) and a
spatio-temporal Graph Attention + LSTM autoencoder (`method="gat"`).

For GAT, each bipolar EEG derivation is a graph node. Two nodes are connected when their bipolar
derivations share an electrode; every node also has a self-loop. Each node's features are that
channel's SWT band energies (7 bands x `swt_steps` time blocks). Two
graph-attention layers learn spatial relationships at each 5-second window. An LSTM then models
the 12-window temporal sequence and reconstructs the graph features.

Both autoencoders are trained only on interictal sequences and use the same reference-error scoring
and seizure-wise alarm evaluation. GAT uses a lower training-sequence cap because graph tensors
require more memory.


In [ ]:
def check_gpu(benchmark=True):
    """Print what TensorFlow can see and, optionally, time a matmul on CPU vs GPU."""
    import platform
    import time
    print(f"TensorFlow {tf.__version__}, built with CUDA: {tf.test.is_built_with_cuda()}")
    gpus = tf.config.list_physical_devices("GPU")
    if not gpus:
        print("No GPU visible to TensorFlow, training will run on CPU.")
        if platform.system() == "Windows":
            print("Note: native Windows GPU support ended with TensorFlow 2.10. Use WSL2 "
                  "(pip install tensorflow[and-cuda]) or tensorflow==2.10 with CUDA 11.2 / cuDNN 8.1.")
        return False
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass   # already set (GPU setup cell) or GPU already in use
        details = tf.config.experimental.get_device_details(gpu)
        print(f"GPU found: {gpu.name} {details.get('device_name', '')} "
              f"(compute capability {details.get('compute_capability', '?')})")
    if benchmark:
        a = tf.random.normal((4000, 4000))
        for dev in ["/CPU:0", "/GPU:0"]:
            with tf.device(dev):
                tf.matmul(a, a).numpy()
                t0 = time.perf_counter()
                for _ in range(10):
                    r = tf.matmul(a, a)
                r.numpy()
                print(f"  10 x 4000x4000 matmul on {dev}: {time.perf_counter() - t0:.2f} s")
    return True


def build_model(seq_len, n_feat, units=64, latent=32, he_leaky=False):
    """he_leaky: He-normal init + LeakyReLU(0.1) on the input Dense layer (the only ReLU layer).
    The LSTMs keep their defaults (tanh, Glorot input / orthogonal recurrent weights), which keeps the
    fast cuDNN kernel; the output Dense stays linear (it rebuilds standardised features)."""
    inp = keras.Input((seq_len, n_feat))
    if he_leaky:
        x = layers.TimeDistributed(layers.Dense(128, kernel_initializer="he_normal"))(inp)
        x = layers.LeakyReLU(0.1)(x)
    else:
        x = layers.TimeDistributed(layers.Dense(128, activation="relu"))(inp)
    x = layers.LSTM(units, return_sequences=True)(x)
    z = layers.LSTM(latent)(x)                              # bottleneck
    x = layers.RepeatVector(seq_len)(z)
    x = layers.LSTM(latent, return_sequences=True)(x)
    x = layers.LSTM(units, return_sequences=True)(x)
    out = layers.TimeDistributed(layers.Dense(n_feat))(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return model


def recon_error(model, S):
    return np.mean((model.predict(S, batch_size=256, verbose=0) - S) ** 2, axis=(1, 2))


def channel_graph_adjacency():
    """Connect bipolar-channel nodes when their derivations share an electrode; include self-loops."""
    n = len(CHANNELS)
    A = np.eye(n, dtype=np.float32)
    endpoints = [set(ch.split("-")) for ch in CHANNELS]
    for i in range(n):
        for j in range(i + 1, n):
            if endpoints[i] & endpoints[j]:
                A[i, j] = A[j, i] = 1.0
    return A


def graph_feature_indices(steps=None, corr=None):
    """Map the flat features to 18 graph nodes: node = channel, its 7 bands x steps SWT values
    (+ its 18 correlations with corr_features)."""
    active = feature_names(steps, corr)
    global_idx = [i for i, n in enumerate(active) if "|" not in n]     # none with SWT features
    groups = []
    for ch in CHANNELS:
        local = [i for i, n in enumerate(active) if n.startswith(ch + "|")]
        groups.append(local + global_idx)
    widths = {len(g) for g in groups}
    if len(widths) != 1 or not groups[0]:
        raise ValueError(f"Invalid graph feature layout: {sorted(widths)}")
    return np.asarray(groups, dtype=np.int32)


def to_graph_sequences(Z_flat, node_indices):
    return np.take(Z_flat, node_indices, axis=2).astype(np.float32)


@keras.utils.register_keras_serializable(package="EEG")
class GraphAttention(layers.Layer):
    """Multi-head graph attention for tensors shaped (..., nodes, features)."""
    def __init__(self, adjacency, units=16, heads=4, dropout=0.1, concat=True, **kwargs):
        super().__init__(**kwargs)
        self.adjacency_config = np.asarray(adjacency, dtype=np.float32).tolist()
        self.units, self.heads = int(units), int(heads)
        self.dropout_rate, self.concat = float(dropout), bool(concat)
        self.attn_dropout = layers.Dropout(self.dropout_rate)

    def build(self, input_shape):
        fin = int(input_shape[-1])
        self.kernel = self.add_weight(name="kernel", shape=(self.heads, fin, self.units),
                                      initializer="glorot_uniform", trainable=True)
        self.attn_src = self.add_weight(name="attn_src", shape=(self.heads, self.units),
                                        initializer="glorot_uniform", trainable=True)
        self.attn_dst = self.add_weight(name="attn_dst", shape=(self.heads, self.units),
                                        initializer="glorot_uniform", trainable=True)
        out_dim = self.heads * self.units if self.concat else self.units
        self.bias = self.add_weight(name="bias", shape=(out_dim,), initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, inputs, training=None):
        A = tf.constant(self.adjacency_config, dtype=inputs.dtype)
        h = tf.einsum("...nf,hfo->...nho", inputs, self.kernel)
        src = tf.einsum("...nho,ho->...nh", h, self.attn_src)
        dst = tf.einsum("...nho,ho->...nh", h, self.attn_dst)
        logits = tf.nn.leaky_relu(src[..., :, None, :] + dst[..., None, :, :], alpha=0.2)
        mask = A[None, None, :, :, None] > 0
        logits = tf.where(mask, logits, tf.cast(-1e9, logits.dtype))
        alpha = tf.nn.softmax(logits, axis=-2)
        alpha = self.attn_dropout(alpha, training=training)
        out = tf.einsum("...ijh,...jho->...iho", alpha, h)
        if self.concat:
            out = tf.reshape(out, tf.concat([tf.shape(out)[:-2], [-1]], axis=0))
        else:
            out = tf.reduce_mean(out, axis=-2)
        return out + self.bias

    def get_config(self):
        cfg = super().get_config()
        cfg.update(dict(adjacency=self.adjacency_config, units=self.units, heads=self.heads,
                        dropout=self.dropout_rate, concat=self.concat))
        return cfg


def build_gat_lstm_autoencoder(seq_len, node_features, adjacency, units=64, latent=32):
    """GAT learns channel relationships per window; LSTM learns temporal evolution."""
    n_nodes = len(CHANNELS)
    inp = keras.Input((seq_len, n_nodes, node_features), name="eeg_graph_sequence")
    x = GraphAttention(adjacency, units=16, heads=4, dropout=0.15, concat=True, name="gat_1")(inp)
    x = layers.ELU()(x)
    x = GraphAttention(adjacency, units=16, heads=2, dropout=0.15, concat=True, name="gat_2")(x)
    x = layers.ELU()(x)
    x = layers.TimeDistributed(layers.Flatten())(x)
    x = layers.LSTM(units, return_sequences=True)(x)
    z = layers.LSTM(latent, name="temporal_bottleneck")(x)
    x = layers.RepeatVector(seq_len)(z)
    x = layers.LSTM(units, return_sequences=True)(x)
    x = layers.TimeDistributed(layers.Dense(n_nodes * node_features))(x)
    out = layers.Reshape((seq_len, n_nodes, node_features), name="reconstructed_eeg_graph")(x)
    model = keras.Model(inp, out, name="GAT_LSTM_autoencoder")
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return model


def graph_recon_error(model, S):
    pred = model.predict(S, batch_size=64, verbose=0)
    return np.mean((pred - S) ** 2, axis=(1, 2, 3))


def fit_baseline(method, Z, args):
    """Returns error(Z) -> one anomaly value per row; larger = more unusual."""
    if method == "pca":
        p = PCA(min(args.baseline_pca_dim, Z.shape[1] - 1, len(Z) - 1), random_state=0).fit(Z)
        return lambda A: ((A - p.inverse_transform(p.transform(A))) ** 2).mean(1)
    if method == "mahalanobis":
        return LedoitWolf().fit(Z).mahalanobis
    if method == "iforest":
        f = IsolationForest(n_estimators=200, random_state=0, n_jobs=-1).fit(Z)
        return lambda A: -f.score_samples(A)
    raise ValueError(method)


def position_features(t, pid=None, gap_s=60.0):
    """Where each window sits in its session, with no EEG: hours since the patient's first window and
    log minutes since the start of the current continuous stretch (a gap > gap_s starts a new one).
    The control from old_work_corrige_v4 (there: run number and time in the run, AUC 0.70)."""
    p = np.zeros(len(t), np.int64) if pid is None else np.asarray(pid, np.int64)
    o = np.lexsort((t, p))
    ts, ps = t[o], p[o]
    new = np.r_[True, (np.diff(ps) != 0) | (np.diff(ts) > gap_s * 1e9)]
    first_of_patient = np.r_[True, np.diff(ps) != 0]
    seg_start = ts[np.maximum.accumulate(np.where(new, np.arange(len(ts)), 0))]
    pat_start = ts[np.maximum.accumulate(np.where(first_of_patient, np.arange(len(ts)), 0))]
    out = np.empty((len(t), 2))
    out[o, 0] = (ts - pat_start) / 3600e9
    out[o, 1] = np.log1p((ts - seg_start) / 60e9)
    return out


def clock_features(t):
    """sin/cos of the clock time of day (timestamps in ns)."""
    a = 2 * np.pi * ((t / 1e9) % 86400) / 86400
    return np.c_[np.sin(a), np.cos(a)]


def inner_split(I, t, args, pid=None):
    """Split training sequences into fit / early-stopping / reference sets.
    interleaved: 15 min blocks in rotation (15 fit, 2 early stop, 3 reference out of every 20),
    so the reference covers every part of the day the model was trained on.
    contiguous: earliest 75% / next 10% / latest 15% in time."""
    te = t[I[:, -1]]
    pe = np.zeros(len(te), np.int64) if pid is None else pid[I[:, -1]].astype(np.int64)
    if args.split == "interleaved":
        blk = np.empty(len(te), np.int64)
        for q in np.unique(pe):                              # blocks counted within each patient
            m = pe == q
            blk[m] = (te[m] - te[m].min()) // int(INNER_BLOCK_MIN * 60e9)
        _, b = np.unique(pe * 10 ** 9 + blk, return_inverse=True)
        r = b % 20
        fit, val, ref = r >= 5, (r == 3) | (r == 4), r < 3
        if fit.sum() >= 10 and ref.sum() >= 10:
            return I[fit], I[val], I[ref]
    a, c = np.quantile(te, [FIT_FRAC, FIT_FRAC + VAL_FRAC])
    return I[te <= a], I[(te > a) & (te <= c)], I[te > c]


def near(tq, tref, gap_ns):
    """True where a query time is within gap_ns of any reference time (tref sorted)."""
    if len(tref) == 0 or gap_ns <= 0:
        return np.zeros(len(tq), bool)
    i = np.searchsorted(tref, tq)
    left, right = tref[np.clip(i - 1, 0, len(tref) - 1)], tref[np.clip(i, 0, len(tref) - 1)]
    return (np.abs(tq - left) <= gap_ns) | (np.abs(right - tq) <= gap_ns)


def interictal_folds(y, t, k, args):
    """Fold number of every interictal window (-1 elsewhere).
    interleaved: --block-min blocks of interictal time assigned to the k folds in rotation,
    so every fold trains on every part of the day. contiguous: k consecutive time blocks."""
    fold = np.full(len(y), -1)
    inter = np.where(y == 0)[0]
    inter = inter[np.argsort(t[inter])]
    if args.split == "interleaved":
        _, b = np.unique((t[inter] - t[inter].min()) // int(args.block_min * 60e9), return_inverse=True)
        fold[inter] = b % k
    else:
        for j, blk in enumerate(np.array_split(inter, k)):
            fold[blk] = j
    return fold


def evenly(I, n):
    """At most n rows of I, spread evenly over its (patient, time) order."""
    return I if len(I) <= n else I[np.linspace(0, len(I) - 1, n).astype(int)]


def fit_supervised(method, X, y, t, train_mask, test_index, args, pid=None):
    """Supervised scorer trained on preictal AND interictal training windows (the autoencoders only see
    interictal data). logreg / gbm: EEG features; posclock: position in the session + clock time, no EEG;
    logreg_posclock: EEG + position + clock. Comparing logreg_posclock with posclock shows whether the EEG
    adds anything to what position and time already give. Scores the last window of each test sequence
    (same scored windows as every other method). Windows are capped at max_train_windows; gbm keeps
    at most 3 interictal windows per preictal window."""
    use_eeg = method in ("logreg", "gbm", "logreg_posclock")
    use_pos = method in ("posclock", "logreg_posclock")
    P = np.c_[position_features(t, pid), clock_features(t)] if use_pos else None

    def feats(idx):
        parts = ([X[idx]] if use_eeg else []) + ([P[idx]] if use_pos else [])
        return np.hstack(parts).astype(np.float32)

    tr = np.flatnonzero(train_mask)
    pre, inter = tr[y[tr] == 1], tr[y[tr] == 0]
    cap = getattr(args, "max_train_windows", 100000)
    pre = evenly(pre, cap // 2)
    if method == "gbm":     # class_weight makes HistGradientBoosting very slow: balance by subsampling instead
        tr = np.r_[pre, evenly(inter, min(3 * len(pre), cap - len(pre)))]
    else:
        tr = np.r_[pre, evenly(inter, cap - len(pre))]
    if (y[tr] == 1).sum() < MIN_TRAIN_PER_CLASS or (y[tr] == 0).sum() < MIN_TRAIN_PER_CLASS:
        raise ValueError("too few training windows of one class for a supervised model")
    sc = StandardScaler().fit(feats(tr))

    def Z(idx):
        return np.clip(sc.transform(feats(idx)), -5, 5)
    if method == "gbm":
        clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                             l2_regularization=1.0, early_stopping=True,
                                             n_iter_no_change=20, random_state=args.seed)
    else:
        clf = LogisticRegression(C=getattr(args, "logreg_C", 0.01) if use_eeg else 1.0,
                                 class_weight="balanced", max_iter=2000)
    clf.fit(Z(tr), y[tr])

    def scorer(mask):
        It = test_index(mask)
        if len(It) == 0:
            return np.zeros(0), np.zeros(0, int), np.zeros(0, np.int64)
        return clf.predict_proba(Z(It[:, -1]))[:, 1], y[It[:, -1]], t[It[:, -1]]
    return scorer


def fit_scorer(method, X, y, g, t, inter_mask, train_mask, args, cfg, pid=None):
    """Fit one method. inter_mask: training interictal windows (autoencoder and anomaly
    baselines); train_mask: all training windows (supervised models and the timeofday / position
    controls).
    Returns (scorer, keras model or None); scorer(mask) -> (0-1 scores, labels, end times)
    for every full sequence inside mask."""
    def test_index(mask):
        return sequence_index(y, g, t, mask, args.seq_len, 1, cfg.win_ns, pid)

    if method in SUPERVISED:
        return fit_supervised(method, X, y, t, train_mask, test_index, args, pid), None

    if method == "timeofday":
        clf = LogisticRegression(class_weight="balanced").fit(clock_features(t[train_mask]), y[train_mask])

        def scorer(mask):
            It = test_index(mask)
            return clf.predict_proba(clock_features(t[It[:, -1]]))[:, 1], y[It[:, -1]], t[It[:, -1]]
        return scorer, None

    if method == "position":
        P = position_features(t, pid)
        sc = StandardScaler().fit(P[train_mask])
        clf = LogisticRegression(class_weight="balanced").fit(sc.transform(P[train_mask]), y[train_mask])

        def scorer(mask):
            It = test_index(mask)
            return clf.predict_proba(sc.transform(P[It[:, -1]]))[:, 1], y[It[:, -1]], t[It[:, -1]]
        return scorer, None

    I = sequence_index(y, g, t, inter_mask, args.seq_len, args.stride, cfg.win_ns, pid)
    if len(I) < 20:
        raise ValueError("too few contiguous interictal training sequences")
    I_fit, I_val, I_ref = inner_split(I, t, args, pid)
    I_fit = evenly(I_fit, args.max_train_seq)                       # memory cap (pooled data is large)
    I_val, I_ref = evenly(I_val, args.max_train_seq // 5), evenly(I_ref, args.max_train_seq // 2)
    scaler = (RobustScaler() if args.scaler == "robust" else StandardScaler()).fit(X[np.unique(I_fit)])

    def Z(I_):                                              # scaled, clipped sequences
        n, L = I_.shape
        return np.clip(scaler.transform(X[I_.reshape(-1)]), -5, 5).reshape(n, L, -1).astype(np.float32)

    def Z_last(I_):                                         # scaled, clipped last windows
        return np.clip(scaler.transform(X[I_[:, -1]]), -5, 5).astype(np.float32)

    model = None
    graph_indices = None
    if method in ("lstm", "gat"):
        keras.backend.clear_session()
        tf.keras.utils.set_random_seed(args.seed)
        if method == "lstm":
            S_fit = Z(I_fit)
            model = build_model(args.seq_len, S_fit.shape[-1], args.units, args.latent,
                                getattr(args, "he_leaky", False))
            if len(I_val):
                S_val = Z(I_val)
                model.fit(S_fit, S_fit, validation_data=(S_val, S_val), epochs=args.epochs, batch_size=64, verbose=0,
                          callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
            else:
                model.fit(S_fit, S_fit, epochs=args.epochs, batch_size=64, verbose=0)
            err1 = lambda I_: recon_error(model, Z(I_))
        else:
            graph_indices = graph_feature_indices(args.swt_steps, getattr(args, "corr_features", False))
            graph_cap = min(args.max_train_seq, getattr(args, "gat_max_train_seq", 5000))
            I_fit = evenly(I_fit, graph_cap)
            I_val = evenly(I_val, max(1, graph_cap // 5))
            I_ref = evenly(I_ref, max(1, graph_cap // 2))
            S_fit = to_graph_sequences(Z(I_fit), graph_indices)
            model = build_gat_lstm_autoencoder(args.seq_len, S_fit.shape[-1], channel_graph_adjacency(),
                                               args.units, args.latent)
            callbacks = [keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
            if len(I_val):
                S_val = to_graph_sequences(Z(I_val), graph_indices)
                model.fit(S_fit, S_fit, validation_data=(S_val, S_val), epochs=args.epochs,
                          batch_size=32, verbose=0, callbacks=callbacks)
            else:
                model.fit(S_fit, S_fit, epochs=args.epochs, batch_size=32, verbose=0)
            err1 = lambda I_: graph_recon_error(model, to_graph_sequences(Z(I_), graph_indices))
    else:
        base = fit_baseline(method, Z_last(np.r_[I_fit, I_val]), args)
        err1 = lambda I_: base(Z_last(I_))

    def err(I_):                                            # in chunks, to bound memory
        return np.concatenate([err1(I_[i:i + 8192]) for i in range(0, len(I_), 8192)]) if len(I_) else np.zeros(0)
    ref_err = np.sort(err(I_ref))

    def scorer(mask):
        It = test_index(mask)
        if len(It) == 0:
            return np.zeros(0), np.zeros(0, int), np.zeros(0, np.int64)
        return np.searchsorted(ref_err, err(It)) / len(ref_err), y[It[:, -1]], t[It[:, -1]]
    scorer.parts = dict(scaler=scaler, ref_err=ref_err, graph_indices=graph_indices)    # saved for inference
    return scorer, model

## Random predictor

In [ ]:
def random_sensitivity(fpr_per_h, sop_min, hours=None):
    """Chance that a Poisson alarm process with rate fpr_per_h raises an alarm inside one SOP.
    With hours given, the rate is at least one false alarm per observed interictal time: seeing
    no false alarm does not make the true rate 0, and a rate of 0 would make any predicted
    seizure count as significant (p = 0)."""
    if hours:
        fpr_per_h = max(fpr_per_h, 1.0 / hours)
    return float(1 - np.exp(-max(fpr_per_h, 0.0) * sop_min / 60))


def poisson_binomial_sf(ps, k):
    """P(X >= k) for X = sum of independent Bernoulli(p_i)."""
    dist = np.array([1.0])
    for p in ps:
        dist = np.convolve(dist, [1 - p, p])
    return float(dist[int(k):].sum())

## Evaluation

In [ ]:
def evaluate_patient(X, y, g, t, method, args, cfg):
    """Leave-one-seizure-out on development data; also picks the alarm threshold."""
    seizure_ids = np.unique(g[y == 1])
    if len(seizure_ids) < 2 or (y == 0).sum() == 0:
        return None
    block_of = interictal_folds(y, t, len(seizure_ids), args)
    gap = int(args.gap_min * 60e9)
    used, skipped = [], []
    for k, s in enumerate(seizure_ids):
        test = ((y == 1) & (g == s)) | ((y == 0) & (block_of == k))
        train_inter = (y == 0) & (block_of != k)
        train_inter &= ~near(t, np.sort(t[(y == 0) & (block_of == k)]), gap)   # no EEG right next to test
        scorer, _ = fit_scorer(method, X, y, g, t, train_inter, ~test & ((y == 1) | train_inter), args, cfg)
        p, yt, tt = scorer(test)
        if len(np.unique(yt)) < 2:
            print(f"  fold {k + 1}/{len(seizure_ids)}: skipped, no full sequence in one class")
            skipped.append(int(s))
            continue
        auc = roc_auc_score(yt, p)
        used.append((int(s), auc, (smooth(p, tt, cfg), yt, tt)))
        print(f"  fold {k + 1}/{len(seizure_ids)}: AUC {auc:.3f}")
    if not used:
        return None
    thr = choose_threshold([fp for _, _, fp in used], cfg)
    folds = []
    for s, auc, (ps, yt, tt) in used:
        det, fa, h = alarm_metrics(ps, yt, tt, thr, cfg)
        folds.append(dict(seizure=s, auc=auc, detected=bool(det), false_alarms=fa, interictal_hours=h))
    f = pd.DataFrame(folds)
    n_det, fa_sum, h_sum = int(f.detected.sum()), float(f.false_alarms.sum()), float(f.interictal_hours.sum())
    fpr = fa_sum / max(h_sum, 1e-9)
    p_rand = random_sensitivity(fpr, cfg.sop_min, h_sum)
    return dict(n_folds=len(f), skipped_folds=";".join(map(str, skipped)), threshold=thr,
                cv_auc_mean=float(f.auc.mean()), cv_detected=n_det, cv_sensitivity=float(f.detected.mean()),
                cv_false_alarms=fa_sum, cv_interictal_h=h_sum, cv_fpr_per_h=fpr,
                cv_random_sensitivity=p_rand, cv_p_vs_random=poisson_binomial_sf([p_rand] * len(f), n_det),
                folds=folds)


def evaluate_holdout(X, y, g, t, dev, hid, thr, method, args, cfg):
    scorer, model = fit_scorer(method, X, y, g, t, dev & (y == 0), dev, args, cfg)
    parts = getattr(scorer, "parts", None)
    p, yt, tt = scorer(hid)
    if len(np.unique(yt)) < 2:
        return None, model, parts
    det, fa, h = alarm_metrics(smooth(p, tt, cfg), yt, tt, thr, cfg)
    det = bool(det)
    fpr = fa / max(h, 1e-9)
    p_rand = random_sensitivity(fpr, cfg.sop_min, h)
    return dict(holdout_auc=float(roc_auc_score(yt, p)), holdout_detected=det, holdout_false_alarms=fa,
                holdout_interictal_h=h, holdout_fpr_per_h=fpr, holdout_random_sensitivity=p_rand,
                holdout_p_vs_random=p_rand if det else 1.0), model, parts


def overall_summary(s, cfg):
    """Pooled and per-patient-mean results for CV and the hidden set, with the random-predictor test."""
    out = {}
    cv = s[s.status.isin(["ok", "cv_only"])]
    if len(cv):
        n, k = int(cv.n_folds.sum()), int(cv.cv_detected.sum())
        ps = np.repeat(cv.cv_random_sensitivity.to_numpy(float), cv.n_folds.to_numpy(int))
        out["cv"] = dict(
            patients=len(cv), seizures=n, predicted=k,
            sensitivity_pooled=k / n, sensitivity_patient_mean=float(cv.cv_sensitivity.mean()),
            fpr_per_h_pooled=float(cv.cv_false_alarms.sum() / max(cv.cv_interictal_h.sum(), 1e-9)),
            fpr_per_h_patient_mean=float(cv.cv_fpr_per_h.mean()),
            auc_patient_mean=float(cv.cv_auc_mean.mean()),
            random_expected_predicted=float(ps.sum()),
            p_vs_random=poisson_binomial_sf(ps, k),
            patients_beating_random=int((cv.cv_p_vs_random < 0.05).sum()))
    ho = s[s.status == "ok"]
    if len(ho):
        k = int(ho.holdout_detected.astype(bool).sum())
        ps = ho.holdout_random_sensitivity.to_numpy(float)
        out["hidden"] = dict(
            patients=len(ho), seizures=len(ho), predicted=k,
            sensitivity_pooled=k / len(ho),
            fpr_per_h_pooled=float(ho.holdout_false_alarms.sum() / max(ho.holdout_interictal_h.sum(), 1e-9)),
            fpr_per_h_patient_mean=float(ho.holdout_fpr_per_h.mean()),
            auc_patient_mean=float(ho.holdout_auc.mean()),
            random_expected_predicted=float(ps.sum()),
            p_vs_random=poisson_binomial_sf(ps, k),
            patients_beating_random=int((ho.holdout_p_vs_random < 0.05).sum()))
    for name, label in (("cv", "Cross-validation"), ("hidden", "Hidden set")):
        if name not in out:
            continue
        o = out[name]
        sens_mean = f", per-patient mean {o['sensitivity_patient_mean']:.2f}" if "sensitivity_patient_mean" in o else ""
        print(f"\n{label}, {o['patients']} patients:\n"
              f"  AUC (per-patient mean)   {o['auc_patient_mean']:.3f}\n"
              f"  seizures predicted       {o['predicted']}/{o['seizures']} "
              f"(sensitivity pooled {o['sensitivity_pooled']:.2f}{sens_mean})\n"
              f"  FPR/h                    pooled {o['fpr_per_h_pooled']:.3f}, "
              f"per-patient mean {o['fpr_per_h_patient_mean']:.3f}\n"
              f"  random predictor         expects {o['random_expected_predicted']:.1f}/{o['seizures']} "
              f"at the same FPR (SOP {cfg.sop_min:g} min), p = {o['p_vs_random']:.3g}\n"
              f"  patients beating random  {o['patients_beating_random']}/{o['patients']} (p < 0.05)")
    return out

## Pooled mode (one model, all patients)

In [ ]:
def save_ae_bundle(args, cfg, name, model, parts, threshold, norms, trained_on, method="lstm"):
    """Save model and inference metadata for the LSTM AE or GAT-LSTM AE."""
    if model is None or not parts or not getattr(args, "save_models", True):
        return
    suffix = "gat_ae" if method == "gat" else "lstm_ae"
    d = os.path.join(args.out, "models")
    os.makedirs(d, exist_ok=True)
    model.save(os.path.join(d, f"{name}_{suffix}.keras"))
    joblib.dump(dict(name=name, method=method, mode=args.mode, threshold=float(threshold), seq_len=args.seq_len,
                     feature_source="swt", swt_steps=args.swt_steps,
                     corr_features=bool(getattr(args, "corr_features", False)),
                     baseline_min=getattr(args, "baseline_min", 0),
                     scaler=parts["scaler"], ref_err=parts["ref_err"], graph_indices=parts.get("graph_indices"),
                     norms=norms, config=asdict(cfg), trained_on=trained_on),
                os.path.join(d, f"{name}_{suffix}.joblib"))
    print(f"saved {os.path.join(d, name + '_' + suffix)}.keras / .joblib")


def norm_stats(X, stats_mask):
    """Median and IQR (std where the IQR is 0) of the stats_mask windows, per feature."""
    A = X[stats_mask].astype(np.float64)
    med = np.median(A, 0)
    q75, q25 = np.percentile(A, [75, 25], axis=0)
    return med, np.where(q75 - q25 > 1e-9, q75 - q25, A.std(0) + 1e-9)


def patient_normalize(X, stats_mask):
    """Robust z-score with one patient's own baseline (median / IQR of stats_mask windows), so
    amplitude differences between patients do not dominate the pooled model."""
    med, scale = norm_stats(X, stats_mask)
    return ((X - med) / scale).astype(np.float32)


def seizure_tests(y, G, t, mask, args):
    """One test mask per seizure inside mask: its preictal windows + an interleaved share of the
    patient's interictal windows in mask."""
    ids = np.unique(G[mask & (y == 1)])
    if len(ids) == 0:
        return []
    fold = interictal_folds(np.where(mask, y, -1), t, len(ids), args)
    return [(int(s), mask & (((y == 1) & (G == s)) | ((y == 0) & (fold == j)))) for j, s in enumerate(ids)]


def score_entries(scorer, tests, names, cfg):
    """Score each (patient, seizure, mask); returns entries with smoothed predictions."""
    out = []
    for pt, s, m in tests:
        p, yt, tt = scorer(m)
        if len(np.unique(yt)) < 2:
            print(f"  {pt} seizure {s % 1000}: skipped, no full sequence in one class")
            continue
        out.append(dict(patient=pt, seizure=s % 1000, auc=float(roc_auc_score(yt, p)),
                        preds=(smooth(p, tt, cfg), yt, tt)))
    return out


def report_entries(entries, thr, cfg, label):
    """Alarm metrics per seizure, per patient and pooled, with the random-predictor test."""
    rows = []
    for e in entries:
        det, fa, h = alarm_metrics(*e["preds"], thr, cfg)
        rows.append(dict(patient=e["patient"], seizure=e["seizure"], auc=e["auc"], detected=bool(det),
                         false_alarms=fa, interictal_hours=h))
    f = pd.DataFrame(rows)
    if f.empty:
        return f, pd.DataFrame(), {}
    per = f.groupby("patient").agg(seizures=("seizure", "size"), predicted=("detected", "sum"),
                                   auc_mean=("auc", "mean"), false_alarms=("false_alarms", "sum"),
                                   interictal_h=("interictal_hours", "sum")).reset_index()
    per["fpr_per_h"] = per.false_alarms / per.interictal_h.clip(lower=1e-9)
    per["random_sensitivity"] = [random_sensitivity(x, cfg.sop_min, h) for x, h in zip(per.fpr_per_h, per.interictal_h)]
    per["p_vs_random"] = [poisson_binomial_sf([r] * int(n), int(k)) for r, n, k in
                          zip(per.random_sensitivity, per.seizures, per.predicted)]
    ps = np.repeat(per.random_sensitivity.to_numpy(float), per.seizures.to_numpy(int))
    n, k = int(per.seizures.sum()), int(per.predicted.sum())
    o = dict(patients=len(per), seizures=n, predicted=k, sensitivity_pooled=k / n,
             sensitivity_patient_mean=float((per.predicted / per.seizures).mean()),
             fpr_per_h_pooled=float(per.false_alarms.sum() / max(per.interictal_h.sum(), 1e-9)),
             fpr_per_h_patient_mean=float(per.fpr_per_h.mean()), auc_seizure_mean=float(f.auc.mean()),
             random_expected_predicted=float(ps.sum()), p_vs_random=poisson_binomial_sf(ps, k),
             patients_beating_random=int((per.p_vs_random < 0.05).sum()), threshold=thr)
    print(f"\n{label}, {o['patients']} patients:\n"
          f"  AUC (mean over seizures) {o['auc_seizure_mean']:.3f}\n"
          f"  seizures predicted       {k}/{n} (sensitivity pooled {o['sensitivity_pooled']:.2f}, "
          f"per-patient mean {o['sensitivity_patient_mean']:.2f})\n"
          f"  FPR/h                    pooled {o['fpr_per_h_pooled']:.3f}, per-patient mean {o['fpr_per_h_patient_mean']:.3f}\n"
          f"  random predictor         expects {o['random_expected_predicted']:.1f}/{n} at the same FPR "
          f"(SOP {cfg.sop_min:g} min), p = {o['p_vs_random']:.3g}\n"
          f"  patients beating random  {o['patients_beating_random']}/{o['patients']} (p < 0.05)")
    return f, per, o


def run_pooled(args, cfg, patients, feat_dir):
    """One model trained on all patients' development data.
    Holdout, never used for training or for choosing the threshold:
      - every training patient's hidden set (last usable seizure + latest 20% of interictal time)
      - whole patients listed in --holdout-patients (all their seizures; the earliest
        --calib-frac of their interictal time is used only to normalise their features)
    The threshold is chosen by patient-grouped cross-validation on the development data:
    --cv-folds groups of patients, each scored by a model trained on the other patients."""
    held_out = set(args.holdout_patients or [])
    Xs, ys, Gs, ts, pids, devs, hids, names = [], [], [], [], [], [], [], []
    norms = {}                                              # per-patient normalisation, saved with the model
    for pt in patients:
        print(f"[{pt}] reading and preprocessing")
        X, y, g, t, info = load_patient(args, pt, cfg, feat_dir)
        n_sz = len(np.unique(g[y == 1]))
        print(f"[{pt}] {int((y == 1).sum())} preictal / {int((y == 0).sum())} interictal windows "
              f"({info['interictal_hours']:.1f} h), {n_sz} usable seizures"
              f"{'  [held-out patient]' if pt in held_out else ''}")
        if (y == 0).sum() < 100:
            print(f"[{pt}] skipped: too little interictal data to normalise")
            continue
        q = len(names)
        if pt in held_out:
            inter = np.where(y == 0)[0]
            inter = inter[np.argsort(t[inter])]
            calib = np.zeros(len(y), bool)
            calib[inter[:int(len(inter) * args.calib_frac)]] = True
            norms[pt] = norm_stats(X, calib)
            Xn, dev, hid = patient_normalize(X, calib), np.zeros(len(y), bool), ~calib
        else:
            split = split_holdout(y, g, t, cfg)
            dev, hid = (np.ones(len(y), bool), np.zeros(len(y), bool)) if split is None else split
            norms[pt] = norm_stats(X, dev & (y == 0))
            Xn = patient_normalize(X, dev & (y == 0))
        Xs.append(Xn); ys.append(y); ts.append(t); pids.append(np.full(len(y), q))
        Gs.append(np.where(g > 0, q * 1000 + g, -1)); devs.append(dev); hids.append(hid); names.append(pt)
    if not names:
        print("no usable patients")
        return
    X, y, G, t, pid, dev, hid = (np.concatenate(a) for a in (Xs, ys, Gs, ts, pids, devs, hids))
    del Xs
    held_idx = [i for i, n in enumerate(names) if n in held_out]
    dev_idx = [i for i, n in enumerate(names) if n not in held_out and (dev & (y == 1) & (pid == i)).any()]
    k = min(args.cv_folds, len(dev_idx))
    print(f"\npooled: {len(names)} patients, {int(dev.sum())} development windows, "
          f"{int((dev & (y == 0)).sum() * cfg.win_s / 3600)} h development interictal, "
          f"{len(dev_idx)} patients with development seizures in {k} CV groups, "
          f"{len(held_idx)} held-out patients")
    if k < 2:
        print("need at least 2 patients with development seizures for cross-validation")
        return
    groups = [dev_idx[j::k] for j in range(k)]
    overall = {}
    for method in args.methods:
        print(f"\n==================== {method} (pooled) ====================")
        cv_entries = []
        for j, grp in enumerate(groups):
            out = np.isin(pid, grp)
            print(f"CV group {j + 1}/{k}: {', '.join(names[i] for i in grp)}")
            scorer, _ = fit_scorer(method, X, y, G, t, dev & ~out & (y == 0), dev & ~out, args, cfg, pid)
            tests = [(names[i], s, m) for i in grp for s, m in seizure_tests(y, G, t, dev & (pid == i), args)]
            cv_entries += score_entries(scorer, tests, names, cfg)
        if not cv_entries:
            print("no CV seizures could be scored")
            continue
        thr = choose_threshold([e["preds"] for e in cv_entries], cfg)
        cv_f, cv_p, cv_o = report_entries(cv_entries, thr, cfg, "Cross-validation (patient groups)")
        print("training the final model on all development data")
        scorer, model = fit_scorer(method, X, y, G, t, dev & (y == 0), dev, args, cfg, pid)
        tag = "lstm_ae" if method == "lstm" else ("gat_ae" if method == "gat" else method)
        if model is not None:
            model.save(os.path.join(args.out, f"pooled_{tag}.keras"))
            save_ae_bundle(args, cfg, "pooled", model, getattr(scorer, "parts", None), thr, norms,
                           "development data of all patients (hidden sets and held-out patients kept out)", method=method)
        tests = []
        for i, n in enumerate(names):
            if i in held_idx:
                tests += [(n, s, m) for s, m in seizure_tests(y, G, t, hid & (pid == i), args)]
            elif (hid & (pid == i)).any():
                tests.append((n, int(np.unique(G[hid & (pid == i) & (y == 1)])[0]), hid & (pid == i)))
        ho_f, ho_p, ho_o = report_entries(score_entries(scorer, tests, names, cfg), thr, cfg,
                                          "HOLDOUT (hidden sets + held-out patients)")
        for f, name in ((cv_f, "cv_seizures"), (cv_p, "cv_patients"), (ho_f, "holdout_seizures"),
                        (ho_p, "holdout_patients")):
            if len(f):
                f.to_csv(os.path.join(args.out, f"pooled_{tag}_{name}.csv"), index=False)
        overall[method] = dict(cv=cv_o, holdout=ho_o)
        with open(os.path.join(args.out, "pooled_overall.json"), "w") as fh:
            json.dump(overall, fh, indent=2)

## Sanity checks (from old_work_corrige_v4)
Run these before the experiment; they only read EDF headers, `events.tsv` and the feature cache.

- **Channel audit**: for every recording, whether the 18 bipolar channels are in the file, rebuilt from a referential montage (A − B), or missing (then the recording is skipped).
- **Label check**: `check_labels` runs on every recording during feature extraction (`check_labels=True` in the settings) and stops with an error if a preictal window ends inside the 5 min horizon, starts before the preictal period, overlaps a seizure or belongs to a non-leading seizure, or if an interictal window is closer than 4 h to any seizure.
- **Label timeline**: one row per recording, windows coloured by label, seizures in black, so you can see the labelling for a patient at a glance.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch


def channel_audit(root, patients):
    """One row per recording: channels native / rebuilt (A - B) / missing, and whether it is used."""
    rows = []
    for pt in patients:
        for rec in build_catalog(root, pt):
            if not rec["has_edf"]:
                rows.append(dict(patient=pt, recording=rec["name"], native=0, rebuilt=0, missing=len(CHANNELS),
                                 used=False, note="EDF missing"))
                continue
            native = sum(len(v) == 1 for v in rec["plan"].values())
            missing = [c for c in CHANNELS if c not in rec["plan"]]
            rows.append(dict(patient=pt, recording=rec["name"], native=native, rebuilt=rec["derived"],
                             missing=len(missing), used=rec["channels_ok"],
                             note=("missing " + ", ".join(missing)) if missing else
                                  ("referential montage rebuilt" if rec["derived"] else "")))
    a = pd.DataFrame(rows)
    print(f"{len(a)} recordings, {int(a.used.sum())} used, {int((a.rebuilt > 0).sum())} rebuilt from a referential "
          f"montage, {int((~a.used).sum())} skipped")
    flagged = a[(a.rebuilt > 0) | ~a.used]
    if len(flagged):
        display(flagged)
    return a


def plot_label_timeline(root, patient, cfg=None, feat_dir=None):
    """Windows of one patient over wall-clock time, one row per recording: interictal blue, preictal red,
    seizures black (leading seizures solid, non-leading hatched), recording span grey."""
    cfg = cfg or make_config(args)
    feat_dir = feat_dir or getattr(args, "feature_cache_dir", None) or os.path.join(args.out, "features")
    catalog = [r for r in build_catalog(root, patient) if r["has_edf"]]
    seizures = seizure_events(build_catalog(root, patient), cfg)
    X, y, g, t, info = load_patient(args, patient, cfg, feat_dir)
    fig, ax = plt.subplots(figsize=(14, 0.32 * len(catalog) + 1.5))
    for k, rec in enumerate(catalog):
        ax.barh(k, rec["end"] - rec["start"], left=rec["start"], height=0.8,
                color="#E5E7EB" if rec["channels_ok"] else "#FDE68A")
        m = (t >= rec["start"].value) & (t < rec["end"].value)
        for lab, col in ((0, "#2563EB"), (1, "#DC2626")):
            tt = pd.to_datetime(t[m & (y == lab)])
            ax.scatter(tt, np.full(len(tt), k), s=2, color=col, marker="|")
        for s in seizures:
            on, end = pd.Timestamp(s["onset"]), pd.Timestamp(s["end"])
            if rec["start"] <= on < rec["end"]:
                ax.axvspan(on, max(end, on + pd.Timedelta(seconds=60)), ymin=0, ymax=1, color="black",
                           alpha=0.8 if s["leading"] else 0.3, hatch=None if s["leading"] else "//")
    ax.set_yticks(range(len(catalog)))
    ax.set_yticklabels([r["name"].replace(f"sub-{patient}_", "").replace("_eeg.edf", "") for r in catalog], fontsize=7)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %H:%M"))
    ax.legend(handles=[Patch(color="#2563EB", label="interictal"), Patch(color="#DC2626", label="preictal"),
                       Patch(color="black", label="leading seizure"), Patch(color="black", alpha=0.3, label="other seizure"),
                       Patch(color="#FDE68A", label="recording skipped (channels)")],
              loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=5, fontsize=8, frameon=False)
    n_pre, n_int = int((y == 1).sum()), int((y == 0).sum())
    ax.set_title(f"{patient}: {n_pre} preictal / {n_int} interictal windows, "
                 f"{info.get('usable_seizures', '?')} usable seizures")
    fig.tight_layout()
    plt.show()


audit_patients = args.patients or sorted(d[4:] for d in os.listdir(args.root)
                                         if d.startswith("sub-") and os.path.isdir(os.path.join(args.root, d)))
audit = channel_audit(args.root, audit_patients)


In [ ]:
# label timeline for one patient (extracts and caches its features on the first call)
plot_label_timeline(args.root, "chb12")


## Run everything
One full run per SOP in `sop_list`.

In [ ]:
def run(args):
    """The whole experiment: features, then pooled or per-patient training and evaluation.
    Results are printed and written to args.out."""
    if not args.root:
        raise ValueError("set root (the folder that contains sub-chb01 ... sub-chb24)")
    global CHECK_LABELS
    CHECK_LABELS = getattr(args, "check_labels", True)
    cfg = make_config(args)
    print(f"SOP {args.sop_min:g} min: preictal {cfg.preictal_min:g} to {cfg.sph_min:g} min before onset, "
          f"refractory {cfg.refractory_min:g} min -> {args.out}")
    feat_dir = getattr(args, "feature_cache_dir", None) or os.path.join(args.out, "features")
    os.makedirs(feat_dir, exist_ok=True)
    os.makedirs(args.out, exist_ok=True)
    if any(m in args.methods for m in ("lstm", "gat")) and not args.features_only:
        print(f"training device: {DEVICE}")
    patients = args.patients or sorted(d[4:] for d in os.listdir(args.root)
                                       if d.startswith("sub-") and os.path.isdir(os.path.join(args.root, d)))
    unknown = set(args.holdout_patients) - set(patients)
    if unknown:
        raise ValueError(f"holdout_patients not among the patients: {' '.join(sorted(unknown))}")
    if args.mode == "pooled" and not args.features_only:
        run_pooled(args, cfg, patients, feat_dir)
        with open(os.path.join(args.out, "config.json"), "w") as fh:
            json.dump(dict(asdict(cfg), **vars(args)), fh, indent=2)
        return
    summary, overview = [], []
    for pt in patients:
        print(f"[{pt}] reading and preprocessing")
        X, y, g, t, info = load_patient(args, pt, cfg, feat_dir)
        n_sz = len(np.unique(g[y == 1]))
        print(f"[{pt}] {int((y == 1).sum())} preictal / {int((y == 0).sum())} interictal windows "
              f"({info['interictal_hours']:.1f} h), {info['seizures']} seizures, {info['leading_seizures']} leading, "
              f"{n_sz} usable")
        overview.append(dict(patient=pt, **{k: v for k, v in info.items() if k not in ("config", "feature_version")}))
        pd.DataFrame(overview).to_csv(os.path.join(args.out, "dataset_overview.csv"), index=False)
        if args.features_only:
            continue
        split = split_holdout(y, g, t, cfg)
        dev = np.ones(len(y), bool) if split is None else split[0]
        if split is None:
            print(f"[{pt}] fewer than 3 usable seizures: no hidden set, cross-validation only")
        for method in args.methods:
            print(f"[{pt}] {method}")
            row = dict(patient=pt, method=method, usable_seizures=n_sz)
            try:
                res = evaluate_patient(X[dev], y[dev], g[dev], t[dev], method, args, cfg)
            except ValueError as e:
                print(f"[{pt}] {method}: {e}")
                res = None
            if res is None:
                print(f"[{pt}] {method} skipped: needs >= 2 usable leading seizures with full sequences")
                row["status"] = "skipped"
            else:
                row.update({k: v for k, v in res.items() if k != "folds"}, status="cv_only")
                name = "lstm_ae" if method == "lstm" else ("gat_ae" if method == "gat" else method)
                pd.DataFrame(res["folds"]).to_csv(os.path.join(args.out, f"{pt}_{name}_cv_folds.csv"), index=False)
                print(f"[{pt}] {method} CV: AUC {res['cv_auc_mean']:.3f}  predicted {res['cv_detected']}/{res['n_folds']}  "
                      f"FPR/h {res['cv_fpr_per_h']:.3f}  (threshold {res['threshold']:.3f})  random would predict "
                      f"{res['cv_random_sensitivity'] * res['n_folds']:.2f}, p = {res['cv_p_vs_random']:.3g}")
                if split is not None:
                    h, model, parts = evaluate_holdout(X, y, g, t, dev, split[1], res["threshold"], method, args, cfg)
                    if model is not None:
                        model.save(os.path.join(args.out, f"{pt}_{name}.keras"))
                        save_ae_bundle(args, cfg, pt, model, parts, res["threshold"], None,
                                       "development data (hidden set kept out)", method=method)
                    if h is None:
                        print(f"[{pt}] {method} HIDDEN: no full sequence in one class, not scored")
                    else:
                        row.update(h, status="ok")
                        print(f"[{pt}] {method} HIDDEN: AUC {h['holdout_auc']:.3f}  seizure predicted: "
                              f"{h['holdout_detected']}  FPR/h {h['holdout_fpr_per_h']:.3f} over "
                              f"{h['holdout_interictal_h']:.1f} h  (random: {h['holdout_random_sensitivity']:.2f} chance)")
                if split is None and method in ("lstm", "gat") and getattr(args, "save_models", True):
                    try:
                        sc, model = fit_scorer(method, X, y, g, t, y == 0, np.ones(len(y), bool), args, cfg)
                        save_ae_bundle(args, cfg, pt, model, sc.parts, res["threshold"], None,
                                       "all data (no hidden set)", method=method)
                    except ValueError as e:
                        print(f"[{pt}] no final model saved: {e}")
            summary.append(row)
            pd.DataFrame(summary).to_csv(os.path.join(args.out, "lstm_ae_summary.csv"), index=False)
    with open(os.path.join(args.out, "config.json"), "w") as fh:
        json.dump(dict(asdict(cfg), **vars(args)), fh, indent=2)
    if not summary:
        return
    s = pd.DataFrame(summary)
    overall = {}
    for method in args.methods:
        sm = s[s.method == method]
        if sm.status.isin(["ok", "cv_only"]).any():
            print(f"\n==================== {method} ====================")
            overall[method] = overall_summary(sm, cfg)
    with open(os.path.join(args.out, "lstm_ae_overall.json"), "w") as fh:
        json.dump(overall, fh, indent=2)

In [ ]:
for mode in MODES:                     # per-patient, then pooled when mode="both"
    args.mode = mode
    for sop in args.sop_list:
        use_sop(args, sop)
        run(args)
args.mode = MODES[0]
use_sop(args, args.sop_list[0])


## Results
Read back from the files written to `out`, for one SOP: set `show_sop` (default: the last one run).

In [ ]:
import glob

for mode in MODES:
    args.mode = mode
    print(f"\n========== {mode} ==========")
    show_sop = args.sop_list[-1]
    use_sop(args, show_sop)
    print(f"SOP {show_sop} min, {args.out}")

    if args.mode == "pooled":
        path = os.path.join(args.out, "pooled_overall.json")
        if os.path.exists(path):
            overall = json.load(open(path))
            rows = [dict(method=m, part=part, **{k: v for k, v in o.items()})
                    for m, d in overall.items() for part, o in d.items() if o]
            display(pd.DataFrame(rows)[["method", "part", "patients", "seizures", "predicted", "sensitivity_pooled",
                                        "fpr_per_h_pooled", "auc_seizure_mean", "random_expected_predicted",
                                        "p_vs_random", "patients_beating_random", "threshold"]])
        for f in sorted(glob.glob(os.path.join(args.out, "pooled_*_holdout_patients.csv"))):
            print(os.path.basename(f))
            display(pd.read_csv(f))
    else:
        path = os.path.join(args.out, "lstm_ae_summary.csv")
        if os.path.exists(path):
            s = pd.read_csv(path)
            cols = [c for c in ["patient", "method", "status", "usable_seizures", "n_folds", "cv_auc_mean", "cv_detected",
                                "cv_fpr_per_h", "cv_p_vs_random", "holdout_auc", "holdout_detected",
                                "holdout_fpr_per_h", "holdout_p_vs_random"] if c in s]
            display(s[cols])
args.mode = MODES[0]


## Compare SOPs
One row per SOP, method and part (cv / holdout in pooled mode, cv / hidden in per-patient mode).

In [ ]:
def compare_sops(args):
    rows = []
    for sop in args.sop_list:
        out = f"{args.out_base}_{mode_tag(args)}_sop{sop}"
        name = "pooled_overall.json" if args.mode == "pooled" else "lstm_ae_overall.json"
        path = os.path.join(out, name)
        if not os.path.exists(path):
            print(f"SOP {sop}: no {name} in {out}")
            continue
        for m, d in json.load(open(path)).items():
            for part, o in d.items():
                if o:
                    rows.append(dict(sop_min=sop, method=m, part=part, seizures=o["seizures"], predicted=o["predicted"],
                                     sensitivity=o["sensitivity_pooled"], fpr_per_h=o["fpr_per_h_pooled"],
                                     auc=o.get("auc_seizure_mean", o.get("auc_patient_mean")),
                                     random_expected=o["random_expected_predicted"], p_vs_random=o["p_vs_random"]))
    f = pd.DataFrame(rows)
    if not f.empty:
        f = f.sort_values(["method", "part", "sop_min"]).round(3)
        f.to_csv(f"{args.out_base}_{mode_tag(args)}_sop_comparison.csv", index=False)
    return f


for mode in MODES:
    args.mode = mode
    print(f"\n========== {mode} ==========")
    display(compare_sops(args))
args.mode = MODES[0]

## Does the EEG add anything over position and time?
`posclock` knows only where a window sits in the session and the clock time; `logreg_posclock` gets the same plus the EEG features. Per patient and SOP, `gain` is the AUC of `logreg_posclock` minus the AUC of `posclock`. A positive gain across patients (one-sided Wilcoxon signed-rank test, `p_gain`) means the EEG carries preictal information that position and time do not. If the gain is ~0, a good AUC from any EEG model may only be the position shortcut. The other methods' AUCs are shown next to it.

In [ ]:
from scipy.stats import wilcoxon


def patient_aucs(out, mode, part="cv"):
    """patient x method AUC table from one results folder (part: "cv" or "hidden"/"holdout")."""
    if mode == "pooled":
        part = "holdout" if part != "cv" else "cv"
        rows = []
        for f in glob.glob(os.path.join(out, f"pooled_*_{part}_patients.csv")):
            m = os.path.basename(f)[len("pooled_"):-len(f"_{part}_patients.csv")]
            m = {"lstm_ae": "lstm", "gat_ae": "gat"}.get(m, m)
            d = pd.read_csv(f)
            rows += [dict(patient=r.patient, method=m, auc=r.auc_mean) for r in d.itertuples()]
        return pd.DataFrame(rows).pivot_table(index="patient", columns="method", values="auc") if rows else pd.DataFrame()
    path = os.path.join(out, "lstm_ae_summary.csv")
    if not os.path.exists(path):
        return pd.DataFrame()
    s = pd.read_csv(path)
    col = "cv_auc_mean" if part == "cv" else "holdout_auc"
    return s.pivot_table(index="patient", columns="method", values=col) if col in s else pd.DataFrame()


def eeg_vs_position(args, part="cv"):
    rows = []
    for sop in args.sop_list:
        out = f"{args.out_base}_{mode_tag(args)}_sop{sop}"
        a = patient_aucs(out, args.mode, part)
        if a.empty:
            print(f"SOP {sop}: no {part} results in {out}")
            continue
        if not {"posclock", "logreg_posclock"} <= set(a.columns):
            print(f"SOP {sop}: needs the posclock and logreg_posclock methods in {out}")
            continue
        a = a.copy()
        a["gain"] = a["logreg_posclock"] - a["posclock"]
        g = a["gain"].dropna()
        p = wilcoxon(g, alternative="greater").pvalue if len(g) >= 5 and (g != 0).any() else float("nan")
        print(f"SOP {sop} min ({part}): EEG adds over position + time in {int((g > 0).sum())}/{len(g)} patients, "
              f"mean gain {g.mean():+.3f}, p_gain = {p:.3g}")
        display(a.round(3))
        rows.append(dict(sop_min=sop, patients=len(g), gain_positive=int((g > 0).sum()),
                         mean_gain=float(g.mean()), p_gain=float(p)))
    return pd.DataFrame(rows)


for mode in MODES:
    args.mode = mode
    print(f"\n========== {mode} ==========")
    eeg_vs_position(args, "cv")
args.mode = MODES[0]

## Per-patient vs pooled
Run the experiment once with `mode="per-patient"` and once with `mode="pooled"` (results go to `<out_base>_pp_sop<N>` and `<out_base>_pooled_sop<N>` automatically). This cell puts the two side by side for one SOP, for every method, including the `position` and `posclock` controls. `compare_runs(sop=20)` picks another SOP.

In [ ]:
def compare_runs(pp_dir=None, pooled_dir=None, sop=None):
    """Per-patient vs pooled for one SOP (default: the first in sop_list), from the folders run() wrote."""
    sop = sop or args.sop_list[0]
    pp_dir = pp_dir or f"{args.out_base}_pp_sop{sop}"
    pooled_dir = pooled_dir or f"{args.out_base}_pooled_sop{sop}"
    rows = []
    p = os.path.join(pp_dir, "lstm_ae_overall.json")
    if os.path.exists(p):
        for m, d in json.load(open(p)).items():
            for part, o in d.items():
                rows.append(dict(run="per-patient", method=m, part=part, patients=o["patients"], seizures=o["seizures"],
                                 predicted=o["predicted"], sensitivity=o["sensitivity_pooled"],
                                 fpr_per_h=o["fpr_per_h_pooled"], auc=o.get("auc_patient_mean"),
                                 p_vs_random=o["p_vs_random"], beating_random=o["patients_beating_random"]))
    p = os.path.join(pooled_dir, "pooled_overall.json")
    if os.path.exists(p):
        for m, d in json.load(open(p)).items():
            for part, o in d.items():
                if o:
                    rows.append(dict(run="pooled", method=m, part=part, patients=o["patients"], seizures=o["seizures"],
                                     predicted=o["predicted"], sensitivity=o["sensitivity_pooled"],
                                     fpr_per_h=o["fpr_per_h_pooled"], auc=o.get("auc_seizure_mean"),
                                     p_vs_random=o["p_vs_random"], beating_random=o["patients_beating_random"]))
    if not rows:
        print("no results yet: run the experiment in per-patient and pooled mode first")
        return None
    out = pd.DataFrame(rows).sort_values(["part", "method", "run"])
    display(out.round(3))
    return out


comparison = compare_runs()


## Visualisations

**Results heatmaps.** Every patient (rows) against every method (columns) for one SOP: AUC, share of seizures predicted and false alarms per hour. A `*` marks a patient where the method beats the random predictor (p < 0.05). `plot_sop_heatmap` puts one method's AUC per patient side by side for all SOPs. The figures are saved in the results folder.


In [ ]:
# ---- results heatmaps: every patient x every method ---------------------------------------
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


def _pp_summary(args, sop):
    path = os.path.join(f"{args.out_base}_pp_sop{sop}", "lstm_ae_summary.csv")
    return pd.read_csv(path) if os.path.exists(path) else None


def _heat(ax, M, title, cmap, norm=None, vmin=None, vmax=None, fmt="{:.2f}", stars=None):
    """One annotated heatmap; stars: same-shape bool frame, '*' where True (beats random)."""
    im = ax.imshow(M.to_numpy(float), aspect="auto", cmap=cmap, norm=norm,
                   vmin=None if norm else vmin, vmax=None if norm else vmax)
    ax.set_xticks(range(M.shape[1]), M.columns, rotation=40, ha="right")
    ax.set_yticks(range(M.shape[0]), M.index)
    ax.set_title(title)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.iat[i, j]
            if pd.isna(v):
                ax.text(j, i, "–", ha="center", va="center", color="0.5", fontsize=8)
                continue
            s = fmt.format(v) + ("*" if stars is not None and bool(stars.iat[i, j]) else "")
            r, gg, b, _ = im.cmap(im.norm(v))
            ax.text(j, i, s, ha="center", va="center", fontsize=8,
                    color="white" if 0.299 * r + 0.587 * gg + 0.114 * b < 0.5 else "black")
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)


def plot_patient_heatmaps(args, sop=None, part="cv", methods=None, save=True):
    """Per-patient mode: patients x methods heatmaps of AUC, sensitivity and false alarms per hour
    for one SOP. part: "cv" or "hidden". * = that patient beats the random predictor (p < 0.05).
    Pooled mode: the AUC heatmap only (from the pooled_*_patients.csv files)."""
    sop = sop or args.sop_list[0]
    if args.mode == "pooled":
        out = f"{args.out_base}_pooled_sop{sop}"
        A = patient_aucs(out, "pooled", part)
        if A.empty:
            print(f"no pooled {part} results in {out}")
            return
        A = A[[m for m in (methods or args.methods) if m in A.columns]]
        fig, ax = plt.subplots(figsize=(1.0 * A.shape[1] + 3, 0.38 * A.shape[0] + 2))
        _heat(ax, A, f"AUC per patient, pooled, SOP {sop} min ({part})", "RdBu_r", TwoSlopeNorm(0.5, 0, 1))
        fig.tight_layout()
        if save:
            fig.savefig(os.path.join(out, f"heatmap_auc_{part}.png"), dpi=150)
        plt.show()
        return A
    s = _pp_summary(args, sop)
    if s is None:
        print(f"no lstm_ae_summary.csv in {args.out_base}_pp_sop{sop}")
        return
    pre = "cv" if part == "cv" else "holdout"
    cols = {"auc": f"{pre}_auc_mean" if pre == "cv" else "holdout_auc",
            "sens": "cv_sensitivity" if pre == "cv" else "holdout_detected",
            "fpr": f"{pre}_fpr_per_h", "p": f"{pre}_p_vs_random"}
    missing = [c for c in cols.values() if c not in s]
    if missing:
        print(f"no {part} columns in the summary ({', '.join(missing)}); hidden sets need >= 3 usable seizures")
        return
    order = [m for m in (methods or args.methods) if m in set(s["method"])]
    piv = lambda c: s.pivot_table(index="patient", columns="method", values=c, aggfunc="first")[order].astype(float)
    auc, sens, fpr, p = piv(cols["auc"]), piv(cols["sens"]), piv(cols["fpr"]), piv(cols["p"])
    stars = (p < 0.05).fillna(False)
    h = 0.38 * len(auc) + 2.2
    fig, axes = plt.subplots(1, 3, figsize=(3 * (0.75 * len(order) + 2.5), h))
    _heat(axes[0], auc, "AUC (0.5 = chance)", "RdBu_r", TwoSlopeNorm(0.5, 0, 1), stars=stars)
    _heat(axes[1], sens, "seizures predicted (share)", "Greens", vmin=0, vmax=1, stars=stars)
    _heat(axes[2], fpr, f"false alarms per hour (limit {args.max_fpr:g})", "Reds",
          vmin=0, vmax=max(1.0, float(np.nanmax(fpr.to_numpy())) if fpr.notna().any().any() else 1.0))
    fig.suptitle(f"Per-patient results, SOP {sop} min, {'cross-validation' if part == 'cv' else 'hidden set'}"
                 f"   (* = beats the random predictor, p < 0.05)")
    fig.tight_layout()
    if save:
        path = os.path.join(f"{args.out_base}_pp_sop{sop}", f"heatmap_{part}.png")
        fig.savefig(path, dpi=150)
        print("saved", path)
    plt.show()
    return dict(auc=auc, sensitivity=sens, fpr_per_h=fpr, p_vs_random=p)


def plot_sop_heatmap(args, method="gat", part="cv", save=True):
    """Per-patient mode: patients x SOP heatmap of one method's AUC (* = beats random)."""
    pre = "cv" if part == "cv" else "holdout"
    ac, pc = (f"{pre}_auc_mean" if pre == "cv" else "holdout_auc"), f"{pre}_p_vs_random"
    auc, p = {}, {}
    for sop in args.sop_list:
        s = _pp_summary(args, sop)
        if s is None or ac not in s:
            continue
        s = s[s["method"] == method].set_index("patient")
        auc[f"SOP {sop}"], p[f"SOP {sop}"] = s[ac].astype(float), s[pc].astype(float)
    if not auc:
        print(f"no per-patient {part} results for {method}")
        return
    A, P = pd.DataFrame(auc), pd.DataFrame(p)
    fig, ax = plt.subplots(figsize=(1.2 * A.shape[1] + 3, 0.38 * len(A) + 2))
    _heat(ax, A, f"{method} AUC per patient and SOP ({part}, * = beats random)", "RdBu_r",
          TwoSlopeNorm(0.5, 0, 1), stars=(P < 0.05).fillna(False))
    fig.tight_layout()
    if save:
        fig.savefig(f"{args.out_base}_pp_heatmap_sop_{method}_{part}.png", dpi=150)
    plt.show()
    return A


for mode in MODES:
    args.mode = mode
    use_sop(args, args.sop_list[0])
    plot_patient_heatmaps(args, part="cv")
    plot_patient_heatmaps(args, part="hidden")
    if mode != "pooled":
        plot_sop_heatmap(args, "gat")
args.mode = MODES[0]
use_sop(args, args.sop_list[0])


**What the GAT learned.** This loads a saved GAT model (`save_models=True`) and computes its attention weights alpha[i, j] (how much channel i listens to channel j) on the patient's interictal and preictal sequences. It draws three figures:
1. the 18 x 18 attention matrices and their difference (preictal minus interictal),
2. the same on a head outline, where edges show attention between linked channels and nodes show how much attention a channel receives,
3. the difference for each head separately.

Small differences are normal, because the model was trained on interictal data only. Look for channels whose difference is large and repeats across patients or SOPs.


In [ ]:
# ---- what the GAT learned: attention between channels ---------------------------------------
# Uses a saved GAT-LSTM model (save_models=True) and the patient's cached features. For each sequence
# the attention weights alpha[i, j] (how much channel i listens to channel j) are computed exactly as
# in GraphAttention.call (no dropout) and averaged over interictal and over preictal sequences.
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import TwoSlopeNorm

# approximate 2-D 10-20 positions (top view, nose up) for drawing the bipolar channels
ELECTRODE_XY = {"FP1": (-0.3, 0.95), "FP2": (0.3, 0.95), "F7": (-0.8, 0.55), "F3": (-0.4, 0.5), "FZ": (0, 0.5),
                "F4": (0.4, 0.5), "F8": (0.8, 0.55), "T7": (-1, 0), "C3": (-0.5, 0), "CZ": (0, 0), "C4": (0.5, 0),
                "T8": (1, 0), "P7": (-0.8, -0.55), "P3": (-0.4, -0.5), "PZ": (0, -0.5), "P4": (0.4, -0.5),
                "P8": (0.8, -0.55), "O1": (-0.3, -0.95), "O2": (0.3, -0.95)}
CHANNEL_XY = np.array([np.mean([ELECTRODE_XY[e] for e in ch.split("-")], 0) for ch in CHANNELS])


def gat_layer_attention(layer, H, batch=256):
    """alpha for inputs H (n, seq, 18, F) -> mean over sequences and time: (18, 18, heads)."""
    W, a_s, a_d = layer.kernel.numpy(), layer.attn_src.numpy(), layer.attn_dst.numpy()
    A = np.asarray(layer.adjacency_config) > 0
    total, n = 0.0, 0
    for k in range(0, len(H), batch):
        h = np.einsum("btnf,hfo->btnho", H[k:k + batch], W)
        src, dst = np.einsum("btnho,ho->btnh", h, a_s), np.einsum("btnho,ho->btnh", h, a_d)
        e = src[:, :, :, None, :] + dst[:, :, None, :, :]
        e = np.where(e > 0, e, 0.2 * e)                                  # LeakyReLU(0.2)
        e = np.where(A[None, None, :, :, None], e, -1e9)
        e = np.exp(e - e.max(axis=3, keepdims=True))
        alpha = e / e.sum(axis=3, keepdims=True)                         # softmax over neighbours j
        total = total + alpha.sum(axis=(0, 1))
        n += alpha.shape[0] * alpha.shape[1]
    return total / max(n, 1)


def gat_attention(patient, sop=None, model_name=None, max_seq=2000):
    """Mean attention of both GAT layers for a patient's interictal and preictal sequences.
    Returns {layer: {"interictal": (18,18,heads), "preictal": (18,18,heads)}}."""
    use_sop(args, sop or args.sop_list[0])
    cfg = make_config(args)
    model_name = model_name or ("pooled" if args.mode == "pooled" else patient)
    base = os.path.join(args.out, "models", f"{model_name}_gat_ae")
    if not os.path.exists(base + ".joblib"):
        raise FileNotFoundError(f"no saved GAT model {base}.keras (run with 'gat' in methods and save_models=True)")
    b = joblib.load(base + ".joblib")
    model = keras.models.load_model(base + ".keras")
    X, y, g, t, info = load_patient(args, patient, cfg, args.feature_cache_dir)
    if b["norms"] is not None:                          # pooled model: the patient's own baseline
        med, scale = b["norms"][patient] if patient in b["norms"] else norm_stats(X, y == 0)
        X = ((X - med) / scale).astype(np.float32)
    gi = b.get("graph_indices")
    gi = graph_feature_indices(b["swt_steps"], b.get("corr_features", False)) if gi is None else gi
    L = b["seq_len"]
    out = {}
    layer_inputs = {name: keras.Model(model.input, model.get_layer(name).input) for name in ("gat_1", "gat_2")}
    for cls, label in ((0, "interictal"), (1, "preictal")):
        I = evenly(sequence_index(y, g, t, y == cls, L, L, cfg.win_ns), max_seq)
        if len(I) == 0:
            print(f"{patient}: no full {label} sequence")
            continue
        Z = np.clip(b["scaler"].transform(X[I.reshape(-1)]), -5, 5).reshape(len(I), L, -1).astype(np.float32)
        S = to_graph_sequences(Z, gi)
        for name, sub in layer_inputs.items():
            H = S if name == "gat_1" else sub.predict(S, batch_size=64, verbose=0)
            out.setdefault(name, {})[label] = gat_layer_attention(model.get_layer(name), H)
        print(f"{patient}: {len(I)} {label} sequences")
    return out


def _scalp(ax, edge_w, node_v, title, cmap, vmax, adjacency):
    """Bipolar channels on a head outline; edges colored by edge_w, nodes by node_v."""
    ax.add_patch(plt.Circle((0, 0), 1.12, fill=False, color="0.6"))
    ax.plot([-0.1, 0, 0.1], [1.11, 1.22, 1.11], color="0.6")
    segs, vals = [], []
    for i in range(len(CHANNELS)):
        for j in range(i + 1, len(CHANNELS)):
            if adjacency[i, j]:
                segs.append([CHANNEL_XY[i], CHANNEL_XY[j]])
                vals.append(edge_w[i, j])
    norm = TwoSlopeNorm(0, -vmax, vmax) if cmap == "RdBu_r" else plt.Normalize(0, vmax)
    lc = LineCollection(segs, cmap=cmap, norm=norm, linewidths=2 + 6 * np.abs(vals) / (vmax or 1))
    lc.set_array(np.asarray(vals))
    ax.add_collection(lc)
    sc = ax.scatter(CHANNEL_XY[:, 0], CHANNEL_XY[:, 1], c=node_v, cmap=cmap, norm=norm, s=260,
                    edgecolors="k", zorder=3)
    for k, ch in enumerate(CHANNELS):
        ax.annotate(ch, CHANNEL_XY[k], xytext=(0, -15), textcoords="offset points", ha="center", fontsize=6.5)
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.35); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=10)
    plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.01)


def plot_gat_attention(patient, sop=None, layer="gat_1", save=True):
    """Three rows of figures for one patient:
    1. attention matrices (mean over heads): interictal, preictal and preictal - interictal
    2. the same on the scalp: edges = attention between linked channels (both directions averaged),
       nodes = attention a channel receives from its neighbours
    3. each head's preictal - interictal matrix, to see if heads specialise."""
    att = gat_attention(patient, sop)
    if layer not in att or len(att[layer]) < 2:
        print("need both interictal and preictal sequences")
        return att
    a0, a1 = att[layer]["interictal"], att[layer]["preictal"]
    m0, m1 = a0.mean(-1), a1.mean(-1)
    d = m1 - m0
    adjacency = np.asarray(channel_graph_adjacency()) > 0
    off = adjacency & ~np.eye(len(CHANNELS), dtype=bool)
    tag = f"{patient}, SOP {sop or args.sop_list[0]} min, {layer}"

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.6))
    vmax = float(max(m0.max(), m1.max()))
    for ax, M, title in ((axes[0], m0, "interictal"), (axes[1], m1, "preictal")):
        im = ax.imshow(np.where(adjacency, M, np.nan), cmap="viridis", vmin=0, vmax=vmax)
        ax.set_title(f"attention, {title}"); plt.colorbar(im, ax=ax, fraction=0.046)
    dv = float(np.abs(d[adjacency]).max()) or 1e-6
    im = axes[2].imshow(np.where(adjacency, d, np.nan), cmap="RdBu_r", norm=TwoSlopeNorm(0, -dv, dv))
    axes[2].set_title("preictal − interictal"); plt.colorbar(im, ax=axes[2], fraction=0.046)
    for ax in axes:
        ax.set_xticks(range(18), CHANNELS, rotation=90, fontsize=7); ax.set_yticks(range(18), CHANNELS, fontsize=7)
        ax.set_xlabel("channel j (listened to)"); ax.set_ylabel("channel i (listening)")
    fig.suptitle(f"GAT attention alpha[i, j], mean over heads — {tag}")
    fig.tight_layout()
    if save:
        fig.savefig(os.path.join(args.out, f"{patient}_{layer}_attention_matrix.png"), dpi=150)
    plt.show()

    sym = lambda M: (M + M.T) / 2
    recv = lambda M: np.where(off, M, 0).sum(0)                      # attention received from neighbours
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    vm = float(max(sym(m0)[off].max(), sym(m1)[off].max(), recv(m0).max(), recv(m1).max()))
    _scalp(axes[0], sym(m0), recv(m0), "interictal", "viridis", vm, adjacency)
    _scalp(axes[1], sym(m1), recv(m1), "preictal", "viridis", vm, adjacency)
    dd = float(max(np.abs(sym(d)[off]).max(), np.abs(recv(m1) - recv(m0)).max())) or 1e-6
    _scalp(axes[2], sym(d), recv(m1) - recv(m0), "preictal − interictal (red = more)", "RdBu_r", dd, adjacency)
    fig.suptitle(f"GAT attention on the scalp — {tag} (self-attention not drawn)")
    fig.tight_layout()
    if save:
        fig.savefig(os.path.join(args.out, f"{patient}_{layer}_attention_scalp.png"), dpi=150)
    plt.show()

    heads = a0.shape[-1]
    fig, axes = plt.subplots(1, heads, figsize=(4.6 * heads, 4.4))
    axes = np.atleast_1d(axes)
    dh = a1 - a0
    hv = float(np.abs(dh[adjacency]).max()) or 1e-6
    for k, ax in enumerate(axes):
        im = ax.imshow(np.where(adjacency, dh[..., k], np.nan), cmap="RdBu_r", norm=TwoSlopeNorm(0, -hv, hv))
        ax.set_title(f"head {k + 1}"); ax.set_xticks([]); ax.set_yticks(range(18), CHANNELS, fontsize=6)
    plt.colorbar(im, ax=axes.tolist(), fraction=0.02)
    fig.suptitle(f"preictal − interictal attention per head — {tag}")
    if save:
        fig.savefig(os.path.join(args.out, f"{patient}_{layer}_attention_heads.png"), dpi=150)
    plt.show()
    return att


# ---- which patient ------------------------------------------------------------------------
viz_patient = "chb01"
attention = {}
for mode in MODES:                     # the per-patient model and the pooled model of that patient
    args.mode = mode
    attention[mode] = plot_gat_attention(viz_patient, sop=args.sop_list[0], layer="gat_1")
args.mode = MODES[0]
use_sop(args, args.sop_list[0])
# for pt in ["chb01", "chb05", "chb10"]:   # several patients
#     plot_gat_attention(pt)


**Channel heatmaps for every patient.** From the cached features, no model needed. For each patient:
1. channel x frequency band: Cohen's d of preictal versus interictal (red = higher before seizures, blue = lower; 0.2 small, 0.5 medium, 0.8 large),
2. channel x minutes before onset: all bands together, z-scored against the patient's interictal windows, averaged over its seizures.

Then one summary over all patients: the strongest change per channel and the band (Hz) it came from. Patients without both classes (such as chb12 with a 4 h margin) are skipped.


In [ ]:
# ---- channel heatmaps for every patient: which channels and bands change before seizures ----
# From the cached SWT features (after the 30-min causal baseline), no model needed.
# Effect size per channel x band: Cohen's d = (mean preictal - mean interictal) / pooled std.
# |d| = 0.2 small, 0.5 medium, 0.8 large; red = higher before seizures, blue = lower.
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

BAND_HZ = ["0-1", "1-2", "2-4", "4-8", "8-16", "16-32", "32-64"]     # cA7, cD7 ... cD2 at 256 Hz


def channel_band_values(X, steps):
    """(n, 18, 7): each window's SWT band value per channel, averaged over its time blocks."""
    names = feature_names(steps, getattr(args, "corr_features", False))
    idx = np.array([[[names.index(f"{ch}|swt_{b}|t{k}") for k in range(steps)] for b in SWT_BANDS]
                    for ch in CHANNELS])                                     # (18, 7, steps)
    return X[:, idx].mean(-1)


def cohens_d(A, B):
    """Effect size of A vs B along axis 0."""
    na, nb = len(A), len(B)
    s = np.sqrt(((na - 1) * A.var(0, ddof=1) + (nb - 1) * B.var(0, ddof=1)) / max(na + nb - 2, 1))
    return (A.mean(0) - B.mean(0)) / np.where(s > 1e-12, s, np.nan)


def seizure_onsets(pt, cfg):
    """{seizure id: onset (ns)} from the recordings' events files."""
    return {s["id"]: s["onset"] for s in seizure_events(build_catalog(args.root, pt), cfg)}


def channel_stats(pt, sop=None, bin_min=2.5):
    """Cohen's d per channel x band, and per channel x minutes-before-onset (all bands averaged,
    z-scored against the patient's interictal windows, mean over its preictal seizures)."""
    use_sop(args, sop or args.sop_list[0])
    cfg = make_config(args)
    X, y, g, t, info = load_patient(args, pt, cfg, args.feature_cache_dir)
    if (y == 1).sum() < 2 or (y == 0).sum() < 2:
        raise ValueError(f"{pt}: needs both preictal and interictal windows "
                         f"({int((y == 1).sum())} / {int((y == 0).sum())})")
    V = channel_band_values(X, args.swt_steps)
    d_band = cohens_d(V[y == 1], V[y == 0])                                  # (18, 7)
    # time course: z-score of the band-averaged value against interictal, per channel
    C = V.mean(-1)                                                           # (n, 18)
    mu, sd = C[y == 0].mean(0), C[y == 0].std(0) + 1e-9
    Zc = (C - mu) / sd
    onsets = seizure_onsets(pt, cfg)
    edges = np.arange(-(cfg.preictal_min), -cfg.sph_min + 1e-9, bin_min)     # e.g. -35 ... -5 min
    rows = []
    for s in np.unique(g[y == 1]):
        m = (y == 1) & (g == s)
        if s not in onsets:
            continue
        mins = (t[m] + cfg.win_ns - onsets[s]) / 60e9                        # window end, minutes to onset
        b = np.digitize(mins, edges) - 1
        rows.append([Zc[m][b == k].mean(0) if (b == k).any() else np.full(18, np.nan)
                     for k in range(len(edges) - 1)])
    z_time = np.nanmean(np.array(rows), axis=0).T if rows else None           # (18, bins)
    return dict(patient=pt, d_band=d_band, z_time=z_time, edges=edges, n_seizures=len(rows),
                n_pre=int((y == 1).sum()), n_inter=int((y == 0).sum()), sop=cfg.preictal_min - cfg.sph_min)


def plot_channel_heatmaps(pt, sop=None, save=True):
    """Two heatmaps for one patient: channel x frequency band (Cohen's d) and channel x minutes
    before onset (z-score against interictal, mean over seizures)."""
    r = channel_stats(pt, sop)
    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), gridspec_kw=dict(width_ratios=[1, 1.4]))
    v = max(0.5, float(np.nanmax(np.abs(r["d_band"]))))
    im = axes[0].imshow(r["d_band"], cmap="RdBu_r", norm=TwoSlopeNorm(0, -v, v), aspect="auto")
    for i in range(18):
        for j in range(7):
            val = r["d_band"][i, j]
            if np.isfinite(val):
                axes[0].text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=7,
                             color="white" if abs(val) > 0.6 * v else "black")
    axes[0].set_xticks(range(7), [f"{b}\n{h} Hz" for b, h in zip(SWT_BANDS, BAND_HZ)], fontsize=8)
    axes[0].set_yticks(range(18), CHANNELS, fontsize=8)
    axes[0].set_title("preictal vs interictal, Cohen's d")
    plt.colorbar(im, ax=axes[0], fraction=0.046)
    if r["z_time"] is not None:
        e = r["edges"]
        zv = max(0.5, float(np.nanmax(np.abs(r["z_time"]))))
        im = axes[1].imshow(r["z_time"], cmap="RdBu_r", norm=TwoSlopeNorm(0, -zv, zv), aspect="auto",
                            extent=[e[0], e[-1], 17.5, -0.5])
        axes[1].set_yticks(range(18), CHANNELS, fontsize=8)
        axes[1].set_xlabel("minutes before seizure onset")
        axes[1].set_title(f"all bands, z-score vs interictal, mean of {r['n_seizures']} seizures")
        plt.colorbar(im, ax=axes[1], fraction=0.046)
    else:
        axes[1].axis("off")
    fig.suptitle(f"{pt}, SOP {r['sop']:g} min: {r['n_pre']} preictal / {r['n_inter']} interictal windows")
    fig.tight_layout()
    if save:
        path = os.path.join(args.out, f"{pt}_channel_heatmaps.png")
        fig.savefig(path, dpi=150)
    plt.show()
    return r


def plot_all_channel_heatmaps(patients=None, sop=None, save=True):
    """plot_channel_heatmaps for every patient, then one summary: patients x channels, largest
    |Cohen's d| over the bands (sign kept)."""
    use_sop(args, sop or args.sop_list[0])
    patients = patients or args.patients or sorted(
        d[4:] for d in os.listdir(args.root) if d.startswith("sub-") and os.path.isdir(os.path.join(args.root, d)))
    summary, which = {}, {}
    for pt in patients:
        try:
            r = plot_channel_heatmaps(pt, sop, save)
        except Exception as ex:
            print(f"{pt} skipped: {ex}")
            continue
        d = r["d_band"]
        k = np.nanargmax(np.abs(np.nan_to_num(d)), axis=1)
        summary[pt] = d[np.arange(18), k]
        which[pt] = [BAND_HZ[i] for i in k]
    if not summary:
        return None
    S = pd.DataFrame(summary, index=CHANNELS).T
    v = max(0.5, float(np.nanmax(np.abs(S.to_numpy()))))
    fig, ax = plt.subplots(figsize=(12, 0.4 * len(S) + 2.5))
    im = ax.imshow(S.to_numpy(float), cmap="RdBu_r", norm=TwoSlopeNorm(0, -v, v), aspect="auto")
    for i, pt in enumerate(S.index):                         # which band (Hz) gave the largest change
        for j in range(18):
            val = S.iat[i, j]
            if np.isfinite(val):
                ax.text(j, i, which[pt][j], ha="center", va="center", fontsize=6,
                        color="white" if abs(val) > 0.6 * v else "black")
    ax.set_xticks(range(18), CHANNELS, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(S)), S.index, fontsize=8)
    ax.set_title(f"strongest preictal change per channel (Cohen's d of the band with the largest |d|, its Hz in the cell), "
                 f"SOP {sop or args.sop_list[0]} min")
    plt.colorbar(im, ax=ax, fraction=0.03)
    fig.tight_layout()
    if save:
        path = os.path.join(args.out, "all_patients_channel_summary.png")
        fig.savefig(path, dpi=150)
        print("saved", path)
    plt.show()
    return S


channel_summary = plot_all_channel_heatmaps(sop=args.sop_list[0])


**Scalp topomaps per band, for every patient.** Top row: Cohen's d of each SWT band, preictal versus interictal, drawn on the head (bipolar channels sit at the midpoint of their two electrodes). Bottom row: where a logistic regression on that patient's SWT features puts its weight, with the share of all weights per band in the title. Saved as `{patient}_channel_topomaps.png`.


In [ ]:
# ---- scalp topomaps per frequency band, for every patient ----------------------------------
# Top row: Cohen's d of each band's SWT value, preictal vs interictal (as in the channel heatmaps).
# Bottom row: where a supervised model looks: share of the absolute logistic-regression weights
# (C = logreg_C, balanced classes, SWT features only) that falls on each channel and band.
# Bipolar channels are drawn at the midpoint of their two electrodes (10-20 positions).
import warnings

import mne
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


def bipolar_info():
    """mne.Info with the 18 bipolar channels placed at the midpoints of their electrodes."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            mon = mne.channels.make_standard_montage("standard_1020")
        except ValueError:
            mon = mne.channels.make_standard_montage("colin27_1020")
    pos = {k.upper(): v for k, v in mon.get_positions()["ch_pos"].items()}
    r = np.mean([np.linalg.norm(pos[e]) for c in CHANNELS for e in c.split("-")])
    ch_pos = {}
    for c in CHANNELS:
        m = np.mean([pos[e] for e in c.split("-")], 0)
        ch_pos[c] = m / np.linalg.norm(m) * r                    # back onto the head surface
    info = mne.create_info(CHANNELS, FS, "eeg")
    info.set_montage(mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame="head"))
    return info


def logreg_weight_share(pt, sop=None):
    """(18, 7) share of |weights| of a logistic regression on the patient's SWT features."""
    use_sop(args, sop or args.sop_list[0])
    cfg = make_config(args)
    X, y, g, t, info = load_patient(args, pt, cfg, args.feature_cache_dir)
    names = feature_names(args.swt_steps, getattr(args, "corr_features", False))
    idx = np.array([[[names.index(f"{ch}|swt_{b}|t{k}") for k in range(args.swt_steps)] for b in SWT_BANDS]
                    for ch in CHANNELS])                                     # (18, 7, steps)
    A = X[:, idx.reshape(-1)]
    clf = LogisticRegression(C=args.logreg_C, class_weight="balanced", max_iter=2000)
    clf.fit(StandardScaler().fit_transform(A), y)
    w = np.abs(clf.coef_[0]).reshape(idx.shape).sum(-1)
    return w / w.sum()


def plot_channel_topomaps(pt, sop=None, save=True):
    """One patient: a row of topomaps of Cohen's d per band, and a row of logistic-regression
    weight share per band."""
    r = channel_stats(pt, sop)
    share = logreg_weight_share(pt, sop)
    info = bipolar_info()
    nb = len(SWT_BANDS)
    fig, axes = plt.subplots(2, nb, figsize=(2.6 * nb + 1.5, 6.4))
    v = max(0.5, float(np.nanmax(np.abs(r["d_band"]))))
    smax = float(share.max())
    for j in range(nb):
        im0, _ = mne.viz.plot_topomap(np.nan_to_num(r["d_band"][:, j]), info, axes=axes[0, j], show=False,
                                      cmap="RdBu_r", vlim=(-v, v), sphere="auto", contours=6)
        axes[0, j].set_title(f"{SWT_BANDS[j]} {BAND_HZ[j]} Hz", fontsize=9)
        im1, _ = mne.viz.plot_topomap(share[:, j], info, axes=axes[1, j], show=False,
                                      cmap="Reds", vlim=(0, smax), sphere="auto", contours=4)
        axes[1, j].set_title(f"{100 * share[:, j].sum():.0f}% of weights", fontsize=9)
    fig.colorbar(im0, ax=axes[0].tolist(), fraction=0.015, pad=0.02, label="Cohen's d (preictal vs interictal)")
    fig.colorbar(im1, ax=axes[1].tolist(), fraction=0.015, pad=0.02, label="share of logreg |weights|")
    fig.suptitle(f"{pt}, SOP {r['sop']:g} min: SWT (db4) band change before seizures (top) "
                 f"and where logistic regression looks (bottom)")
    if save:
        path = os.path.join(args.out, f"{pt}_channel_topomaps.png")
        fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return dict(d_band=r["d_band"], weight_share=share)


def plot_all_channel_topomaps(patients=None, sop=None, save=True):
    use_sop(args, sop or args.sop_list[0])
    patients = patients or args.patients or sorted(
        d[4:] for d in os.listdir(args.root) if d.startswith("sub-") and os.path.isdir(os.path.join(args.root, d)))
    out = {}
    for pt in patients:
        try:
            out[pt] = plot_channel_topomaps(pt, sop, save)
        except Exception as ex:
            print(f"{pt} skipped: {ex}")
    return out


topomaps = plot_all_channel_topomaps(sop=args.sop_list[0])


**Scalp topomaps of mean log band power, Welch and wavelet.** Computed from the raw EEG, because the cached features have the 30-min baseline subtracted. For each patient there are two figures, `{patient}_topomap_logpower_welch.png` (delta to gamma) and `{patient}_topomap_logpower_wavelet.png` (SWT bands). Each has three rows: the interictal mean, the preictal mean, and preictal minus interictal. This reads every recording once per patient, then caches the values in `{patient}_band_power.npz`.


In [ ]:
# ---- scalp topomaps of mean log band power: Welch (classic bands) and wavelet (SWT bands) ----
# Read from the raw EEG (the cached features have the 30-min baseline subtracted, so their
# absolute level is gone). For each patient: all preictal windows and an even sample of
# interictal windows; per window and channel
#   Welch:   log10 mean PSD in delta 1-4, theta 4-8, alpha 8-13, beta 13-30, gamma 30-45 Hz (uV^2/Hz)
#   wavelet: log10 mean squared SWT coefficient (db4, level 7) in cA7, cD7 ... cD2 (uV^2)
# Rows: interictal mean, preictal mean (same color scale per band), preictal - interictal.
# The values are cached in <out>/<patient>_band_power.npz, delete it to recompute.
from scipy.signal import welch

WELCH_BANDS = {"delta": (1, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 45)}


def window_log_power(segs):
    """segs (n, 18, samples) in uV -> Welch (n, 18, 5) and wavelet (n, 18, 7) log10 power."""
    f, P = welch(segs, fs=FS, nperseg=2 * FS, axis=-1)
    welch_lp = np.stack([np.log10(P[..., (f >= lo) & (f < hi)].mean(-1) + 1e-12)
                         for lo, hi in WELCH_BANDS.values()], -1)
    sw = pywt.swt(segs.astype(np.float64), SWT_WAVELET, level=SWT_LEVEL, axis=-1, trim_approx=True)
    wav_lp = np.stack([np.log10((c ** 2).mean(-1) + 1e-12) for c in sw[:SWT_LEVEL]], -1)
    return welch_lp, wav_lp


def patient_band_power(pt, sop=None, inter_per_recording=120, refresh=False):
    """Log band power of a patient's preictal windows and a sample of interictal windows."""
    use_sop(args, sop or args.sop_list[0])
    cfg = make_config(args)
    os.makedirs(args.out, exist_ok=True)
    cache = os.path.join(args.out, f"{pt}_band_power.npz")
    if os.path.exists(cache) and not refresh:
        z = np.load(cache)
        return z["welch"], z["wavelet"], z["y"]
    catalog = build_catalog(args.root, pt)
    seizures = seizure_events(catalog, cfg)
    n = int(cfg.win_s * FS)
    W, V, Y = [], [], []
    for rec in catalog:
        if not (rec["has_edf"] and rec["channels_ok"]):
            continue
        dur = (rec["end"] - rec["start"]).total_seconds()
        ws = pd.DatetimeIndex(rec["start"] + pd.to_timedelta(np.arange(0, dur - cfg.win_s + 1e-9, cfg.win_s),
                                                             unit="s")).as_unit("ns").asi8
        lab, _ = label_windows(ws, seizures, cfg)
        if not (lab >= 0).any():
            continue
        data = load_bipolar(rec["path"], rec["plan"], cfg)
        idx = np.arange(min(len(ws), data.shape[1] // n))
        pre, inter = idx[lab[idx] == 1], idx[lab[idx] == 0]
        if len(inter) > inter_per_recording:
            inter = inter[np.linspace(0, len(inter) - 1, inter_per_recording).astype(int)]
        idx = np.r_[pre, inter]
        idx = idx[windows_ok(data, idx, n, cfg)]
        if len(idx) == 0:
            continue
        segs = np.stack([data[:, i * n:(i + 1) * n] for i in idx])
        w, v = window_log_power(segs)
        W.append(w); V.append(v); Y.append(lab[idx])
        print(f"  {rec['name']}: {int((lab[idx] == 1).sum())} preictal, {int((lab[idx] == 0).sum())} interictal")
    if not W:
        raise ValueError(f"{pt}: no labelled windows")
    W, V, Y = np.concatenate(W), np.concatenate(V), np.concatenate(Y)
    np.savez_compressed(cache, welch=W, wavelet=V, y=Y)
    return W, V, Y


def _topo_rows(P, y, band_labels, title, path):
    """3 rows x bands of topomaps: interictal, preictal, preictal - interictal (log10 units)."""
    info = bipolar_info()
    m0, m1 = P[y == 0].mean(0), P[y == 1].mean(0)                    # (18, bands)
    nb = len(band_labels)
    fig, axes = plt.subplots(3, nb, figsize=(2.5 * nb + 1, 8.6))
    dv = float(np.abs(m1 - m0).max()) or 1e-6
    for j, lab in enumerate(band_labels):
        lo, hi = float(min(m0[:, j].min(), m1[:, j].min())), float(max(m0[:, j].max(), m1[:, j].max()))
        for row, M in ((0, m0), (1, m1)):
            im, _ = mne.viz.plot_topomap(M[:, j], info, axes=axes[row, j], show=False, cmap="viridis",
                                         vlim=(lo, hi), sphere="auto", contours=4)
        cb = fig.colorbar(im, ax=axes[:2, j].tolist(), orientation="horizontal", fraction=0.05, pad=0.04)
        cb.ax.tick_params(labelsize=7)
        imd, _ = mne.viz.plot_topomap(m1[:, j] - m0[:, j], info, axes=axes[2, j], show=False, cmap="RdBu_r",
                                      vlim=(-dv, dv), sphere="auto", contours=4)
        axes[0, j].set_title(lab, fontsize=9)
    for row, name in enumerate(["interictal", "preictal", "preictal − interictal"]):
        axes[row, 0].text(-0.25, 0.5, name, transform=axes[row, 0].transAxes, rotation=90,
                          va="center", ha="center", fontsize=10)
    fig.colorbar(imd, ax=axes[2].tolist(), fraction=0.015, pad=0.02, label="difference (log10 power)")
    fig.suptitle(title)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_band_power_topomaps(pt, sop=None):
    """Mean log band power topomaps for one patient, Welch and wavelet (two figures)."""
    W, V, y = patient_band_power(pt, sop)
    if (y == 1).sum() == 0 or (y == 0).sum() == 0:
        raise ValueError(f"{pt}: needs both preictal and interictal windows")
    sop_min = sop or args.sop_list[0]
    tag = f"{pt}, SOP {sop_min} min: {int((y == 1).sum())} preictal / {int((y == 0).sum())} interictal windows"
    _topo_rows(W, y, [f"{b} {lo}-{hi} Hz" for b, (lo, hi) in WELCH_BANDS.items()],
               f"Mean log10 band power, Welch PSD (uV²/Hz) — {tag}",
               os.path.join(args.out, f"{pt}_topomap_logpower_welch.png"))
    _topo_rows(V, y, [f"{b} {h} Hz" for b, h in zip(SWT_BANDS, BAND_HZ)],
               f"Mean log10 wavelet power, SWT db4 (uV²) — {tag}",
               os.path.join(args.out, f"{pt}_topomap_logpower_wavelet.png"))


def plot_all_band_power_topomaps(patients=None, sop=None):
    use_sop(args, sop or args.sop_list[0])
    patients = patients or args.patients or sorted(
        d[4:] for d in os.listdir(args.root) if d.startswith("sub-") and os.path.isdir(os.path.join(args.root, d)))
    for pt in patients:
        print(f"[{pt}] band power")
        try:
            plot_band_power_topomaps(pt, sop)
        except Exception as ex:
            print(f"{pt} skipped: {ex}")


plot_all_band_power_topomaps(sop=args.sop_list[0])


## Predict on a new recording
Runs a trained model on one EDF file and answers: **when would it have raised an alarm, and when is a seizure expected?** An alarm means *a seizure is likely between 5 and 35 minutes from now* (5 min horizon, 30 min occurrence period). The recording goes through exactly the same preprocessing as in training (18 bipolar channels, 0.5-45 Hz, 5 s windows, artifact rules, SWT features), then the score is smoothed over 1 min and compared with the saved threshold, with a 30 min pause after each alarm.

It loads `<out>/models/pooled_lstm_ae.keras/.joblib` (pooled mode) or `<out>/models/<patient>_lstm_ae...` (per-patient mode), saved by the experiment above when `methods` includes `"lstm"`: the autoencoder, the feature scaling, the reference errors that turn errors into 0-1 scores, the threshold and, in pooled mode, each patient's normalisation. A patient the pooled model never saw is normalised with the recording's own median / IQR.

Output: the alarm list, a plot of the risk score over time, and CSV / PNG files in `<out>/predictions`. If an `events.tsv` sits next to the EDF (CHB-MIT BIDS), annotated seizures are drawn and each alarm is marked correct or false.

**Use recordings the model has not trained on** for an honest check: a file from a patient's hidden set (its last usable seizure, or the latest part of the recordings), a held-out patient, or a new recording. A recording used in training gives optimistic results.

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def recording_start(edf_path, raw):
    """Wall-clock start of a recording: from the BIDS scans.tsv next to it (CHB-MIT), else the EDF
    header, else an arbitrary date (times are then relative to the start of the file)."""
    sub_dir = os.path.dirname(os.path.dirname(os.path.abspath(edf_path)))
    for scans in glob.glob(os.path.join(sub_dir, "*_scans.tsv")):
        s = pd.read_csv(scans, sep="\t", encoding="utf-8-sig")
        hit = s[s["filename"].astype(str).str.replace("\\", "/").str.endswith(os.path.basename(edf_path))]
        if len(hit):
            ts = pd.to_datetime(hit["acq_time"].iloc[0], errors="coerce")
            if not pd.isna(ts):
                return to_naive(ts), "scans.tsv"
    md = raw.info.get("meas_date")
    if md is not None:
        return to_naive(pd.Timestamp(md)), "EDF header"
    return pd.Timestamp("2000-01-01").as_unit("ns"), "unknown (times are relative to the file start)"


def read_recording(edf_path, cfg):
    """Same preprocessing as training: 18 bipolar channels, 256 Hz, 0.5-45 Hz FIR on the whole file,
    5 s windows, artifact rules. Returns the filtered data, the windows, a valid-window mask, window
    start times (ns), the recording start and where that start time came from."""
    raw = mne.io.read_raw_edf(edf_path, preload=False)
    plan = channel_plan(raw.ch_names)
    missing = [c for c in CHANNELS if c not in plan]
    if missing:
        raise ValueError(f"{os.path.basename(edf_path)} lacks channels: {', '.join(missing)}")
    start, source = recording_start(edf_path, raw)
    data = load_bipolar(edf_path, plan, cfg)
    n = int(cfg.win_s * FS)
    nw = data.shape[1] // n
    segs = data[:, :nw * n].reshape(len(CHANNELS), nw, n).transpose(1, 0, 2)
    ok = (np.isfinite(segs).all((1, 2)) & (np.abs(segs).max((1, 2)) <= cfg.max_abs_uv)
          & (segs.std(-1).min(-1) >= cfg.min_std_uv))
    ws = start.value + np.arange(nw, dtype=np.int64) * cfg.win_ns
    return data, segs, ok, ws, start, source


def seizures_in_file(edf_path, start):
    """Annotated seizures (onset, end) of the file, if a BIDS events.tsv sits next to it."""
    ev_path = edf_path.replace("_eeg.edf", "_events.tsv")
    if not os.path.exists(ev_path):
        return []
    ev = pd.read_csv(ev_path, sep="\t", encoding="utf-8-sig")
    if "trial_type" in ev:
        ev = ev[ev["trial_type"].astype(str).str.contains("seizure", case=False)]
    return [(start + pd.to_timedelta(float(e["onset"]), unit="s"),
             start + pd.to_timedelta(float(e["onset"]) + float(e["duration"]), unit="s")) for _, e in ev.iterrows()]


def find_alarms(ps, tt, thr, cfg):
    """Alarm times (ns, end of the window that crossed the threshold) with the 30 min refractory period."""
    o = np.argsort(tt)
    last, out = -np.inf, []
    for i in o[ps[o] >= thr]:
        if tt[i] - last >= cfg.refractory_min * 60e9:
            last = tt[i]
            out.append(tt[i] + cfg.win_ns)
    return out


def report_prediction(edf_path, tt, p, ps, thr, cfg, start, source, n_windows, n_valid, label, trained_on):
    """Print and save the alarms, plot the risk score over time, and return (scores, alarms)."""
    out_dir = os.path.join(args.out, "predictions")
    os.makedirs(out_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(edf_path))[0]
    seizures = seizures_in_file(edf_path, start)
    sph, sop_end = pd.Timedelta(minutes=cfg.sph_min), pd.Timedelta(minutes=cfg.preictal_min)
    rows = []
    for a in find_alarms(ps, tt, thr, cfg):
        a = pd.Timestamp(a)
        lo, hi = a + sph, a + sop_end
        hits = [on for on, _ in seizures if lo < on <= hi]
        rows.append(dict(alarm=a, seizure_expected_from=lo, seizure_expected_until=hi,
                         annotated_seizure_in_window=(hits[0] if hits else None) if seizures else "no annotations"))
    alarms = pd.DataFrame(rows, columns=["alarm", "seizure_expected_from", "seizure_expected_until",
                                         "annotated_seizure_in_window"])
    scores = pd.DataFrame(dict(window_end=pd.to_datetime(tt + cfg.win_ns), score=p, smoothed=ps)).sort_values("window_end")
    scores.to_csv(os.path.join(out_dir, f"{base}_{label}_scores.csv"), index=False)
    alarms.to_csv(os.path.join(out_dir, f"{base}_{label}_alarms.csv"), index=False)

    print(f"{os.path.basename(edf_path)}  [{label}, model trained on {trained_on}]")
    print(f"  start {start:%Y-%m-%d %H:%M:%S} (from {source}), {n_windows} windows of {cfg.win_s:g} s, "
          f"{n_windows - n_valid} rejected as artifacts, {len(scores)} scored, threshold {thr:.3f}")
    if alarms.empty:
        print("  no alarm: no seizure expected in the next 5-35 min at any point of this recording")
    for _, r in alarms.iterrows():
        tail = ""
        if seizures:
            tail = (f"  -> annotated seizure at {r.annotated_seizure_in_window:%H:%M:%S} (correct)"
                    if isinstance(r.annotated_seizure_in_window, pd.Timestamp) else "  -> no seizure in that window (false alarm)")
        print(f"  ALARM at {r.alarm:%H:%M:%S}: seizure expected between {r.seizure_expected_from:%H:%M} "
              f"and {r.seizure_expected_until:%H:%M}{tail}")
    for on, _ in seizures:
        caught = any(lo < on <= hi for lo, hi in zip(alarms.seizure_expected_from, alarms.seizure_expected_until))
        print(f"  annotated seizure at {on:%H:%M:%S}: {'predicted' if caught else 'not predicted'}")

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(scores.window_end, scores.score, color="#9CA3AF", lw=0.6, label="score per window")
    ax.plot(scores.window_end, scores.smoothed, color="#1D4ED8", lw=1.6, label="smoothed (1 min)")
    ax.axhline(thr, color="#DC2626", ls="--", lw=1, label=f"threshold {thr:.3f}")
    for i, r in alarms.iterrows():
        ax.axvline(r.alarm, color="#DC2626", lw=1.2)
        ax.axvspan(r.seizure_expected_from, r.seizure_expected_until, color="#FCA5A5", alpha=0.25,
                   label="seizure expected (alarm + 5-35 min)" if i == 0 else None)
    for k, (on, end) in enumerate(seizures):
        ax.axvspan(on, max(end, on + pd.Timedelta(seconds=30)), color="black", alpha=0.6,
                   label="annotated seizure" if k == 0 else None)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("preictal risk score (0-1)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.set_title(f"{os.path.basename(edf_path)}: {label}, {len(alarms)} alarm(s)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8, ncol=5, frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"{base}_{label}.png"), dpi=120)
    plt.show()
    print(f"  saved to {out_dir}")
    return scores, alarms

In [ ]:
def predict_edf(edf_path, patient, model_name=None, method=None):
    """Score one EDF using a saved GAT-LSTM or LSTM autoencoder.
    model_name: "pooled" or a patient id. method: "gat", "lstm", or None (prefer GAT if saved).
    With a pooled model, features use that patient's stored baseline; unseen patients use the
    recording's own median / IQR."""
    model_name = model_name or ("pooled" if args.mode == "pooled" else patient)
    if method not in (None, "gat", "lstm"):
        raise ValueError("method must be None, 'gat' or 'lstm'")
    suffix = "gat_ae" if method == "gat" else "lstm_ae"
    base = os.path.join(args.out, "models", f"{model_name}_{suffix}")
    if method is None:
        gat_base = os.path.join(args.out, "models", f"{model_name}_gat_ae")
        if os.path.exists(gat_base + ".joblib"):
            base = gat_base
    if not os.path.exists(base + ".joblib"):
        raise FileNotFoundError(f"No saved GAT or LSTM autoencoder bundle found for {model_name}")
    b = joblib.load(base + ".joblib")
    model = keras.models.load_model(base + ".keras")
    cfg_p = Config(**b["config"])
    data, segs, ok, ws, start, source = read_recording(edf_path, cfg_p)
    idx = np.where(ok)[0]
    if b.get("feature_source") != "swt":
        raise ValueError("this model was trained on the old 829 / paper features; retrain it with SWT features")
    steps, corr, base = b["swt_steps"], b.get("corr_features", False), b.get("baseline_min", 0)
    X = np.concatenate([window_features(segs[idx[i:i + 128]], steps, corr) for i in range(0, len(idx), 128)])
    if base:                                                # same causal per-recording baseline as in training
        X, valid = causal_baseline(X, ws[idx], base, cfg_p.win_s)
        X, idx = X[valid], idx[valid]
        print(f"baseline: the first {BASELINE_MIN_PAST_MIN:g} min of the recording (and after gaps in it) have no score")
    if b["norms"] is not None:
        if patient in b["norms"]:
            med, scale = b["norms"][patient]
        else:
            print(f"{patient} was not in the pooled training data: normalising with this recording's own median / IQR")
            med, scale = norm_stats(X, np.ones(len(X), bool))
        X = ((X - med) / scale).astype(np.float32)
    Z = np.clip(b["scaler"].transform(X), -5, 5).astype(np.float32)
    tt0 = ws[idx]
    zeros = np.zeros(len(idx), int)
    I = sequence_index(zeros, zeros - 1, tt0, np.ones(len(idx), bool), b["seq_len"], 1, cfg_p.win_ns)
    if len(I) == 0:
        raise ValueError("no run of consecutive valid windows long enough for one sequence")
    if b.get("method", "lstm") == "gat":
        graph_indices = b.get("graph_indices")
        if graph_indices is None:
            graph_indices = graph_feature_indices(steps, corr)
        err = np.concatenate([
            graph_recon_error(model, to_graph_sequences(Z[I[k:k + 1024]], graph_indices))
            for k in range(0, len(I), 1024)
        ])
    else:
        err = np.concatenate([
            np.mean((model.predict(Z[I[k:k + 4096]], batch_size=256, verbose=0) - Z[I[k:k + 4096]]) ** 2,
                    axis=(1, 2))
            for k in range(0, len(I), 4096)
        ])
    p = np.searchsorted(b["ref_err"], err) / len(b["ref_err"])
    tt = tt0[I[:, -1]]
    ps = smooth(p, tt, cfg_p)
    model_label = "GAT-LSTM graph autoencoder" if b.get("method") == "gat" else "LSTM autoencoder"
    print(f"model: {model_name} {model_label}, SWT features ({X.shape[1]}, {steps} blocks per window), "
          f"sequences of {b['seq_len']} windows; the first {b['seq_len'] - 1} windows after a gap have no score")
    return report_prediction(edf_path, tt, p, ps, b["threshold"], cfg_p, start, source, len(ws), len(idx),
                             f"{b.get('method', 'lstm')}_{model_name}", b["trained_on"])

In [ ]:
# ---- which recording, and which patient's model --------------------------------------
use_sop(args, args.sop_list[0])     # which SOP's saved model to use
predict_patient = "chb01"
predict_edf_path = r"C:\dataset chb\.nemar\sub-chb01\eeg\sub-chb01_task-rest_run-26_eeg.edf"
# chb01 run 26 contains chb01's hidden-set seizure (onset 13:05), never used for training

scores, alarms = predict_edf(predict_edf_path, predict_patient, method="gat")
alarms

## Agent: ask questions about the results (NVIDIA API)

An LLM from build.nvidia.com that reads the saved results and can run predictions with the saved models.
It uses this notebook's functions as tools, so it never trains anything and never makes up numbers.

1. `pip install openai`, and get an API key at https://build.nvidia.com (starts with `nvapi-`).
2. Run the settings cell and the cells that define functions. Skip the "run everything" cell if the results already exist.
3. Run the agent cell. It asks for the key unless `NVIDIA_API_KEY` is set. Don't paste the key into the notebook.
4. Ask questions with `ask_agent(...)`. Pass the returned `chat` back in to keep the conversation going.

Tools: `list_runs`, `overall_results`, `patient_results`, `compare_sops`, `eeg_vs_position`, `list_recordings`, `predict_recording`.


In [ ]:
# ---- seizure-prediction agent (NVIDIA API) ----------------------------------------------
# An LLM that answers questions about the saved results and runs predictions with the saved models.
# It calls the functions of this notebook as tools. Nothing is trained here: run the settings cell and
# the cells that define functions (skip the "run everything" cell), then this cell.
#
# Setup, once:  pip install openai
# API key: get one at https://build.nvidia.com (it starts with "nvapi-"). Do not paste it in the notebook.
# Set it as an environment variable before starting Jupyter:
#     Windows:  setx NVIDIA_API_KEY "nvapi-..."    (then reopen the terminal / restart Jupyter)
# or type it when this cell asks for it.
import contextlib
import getpass
import io

from openai import OpenAI

AGENT_MODEL = "meta/llama-3.3-70b-instruct"     # any model on build.nvidia.com that supports tool calling
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key (nvapi-...): ")
nvidia_client = OpenAI(base_url=NVIDIA_BASE_URL, api_key=os.environ["NVIDIA_API_KEY"])


# ---- tools: plain functions that return text ----------------------------------------------
def _quiet(fn, *a, **k):
    """Run a notebook function without its printed output and tables."""
    with contextlib.redirect_stdout(io.StringIO()):
        g = globals()
        saved = g.get("display")
        g["display"] = lambda *x, **y: None
        try:
            return fn(*a, **k)
        finally:
            g["display"] = saved


def _table(df, max_rows=60):
    if df is None or len(df) == 0:
        return "no rows"
    return df.head(max_rows).round(3).to_string(index=False)


def _out_dir(sop, mode):
    sop = sop or args.sop_list[0]
    tag = "pp" if mode in (None, "per-patient", "pp") else "pooled"
    return f"{args.out_base}_{tag}_sop{int(sop)}"


def tool_list_runs():
    rows = []
    for d in sorted(glob.glob(f"{args.out_base}_*_sop*")):
        if os.path.isdir(d):
            models = sorted(os.listdir(os.path.join(d, "models"))) if os.path.isdir(os.path.join(d, "models")) else []
            rows.append(f"{d}: files {len(os.listdir(d))}, saved models {len(models) // 2}")
    return "\n".join(rows) or f"no result folders matching {args.out_base}_*_sop* (run the experiment first)"


def tool_overall(sop=None, mode="per-patient"):
    out = _out_dir(sop, mode)
    name = "pooled_overall.json" if mode == "pooled" else "lstm_ae_overall.json"
    path = os.path.join(out, name)
    if not os.path.exists(path):
        return f"no {name} in {out}"
    rows = []
    for m, d in json.load(open(path)).items():
        for part, o in d.items():
            if o:
                rows.append(dict(method=m, part=part, patients=o.get("patients"), seizures=o.get("seizures"),
                                 predicted=o.get("predicted"), sensitivity=o.get("sensitivity_pooled"),
                                 fpr_per_h=o.get("fpr_per_h_pooled"),
                                 auc=o.get("auc_patient_mean", o.get("auc_seizure_mean")),
                                 random_expected=o.get("random_expected_predicted"), p_vs_random=o.get("p_vs_random"),
                                 patients_beating_random=o.get("patients_beating_random")))
    return f"{out} ({mode})\n" + _table(pd.DataFrame(rows))


def tool_patient(patient, sop=None, mode="per-patient"):
    out = _out_dir(sop, mode)
    if mode == "pooled":
        rows = []
        for f in sorted(glob.glob(os.path.join(out, "pooled_*_patients.csv"))):
            d = pd.read_csv(f)
            d = d[d["patient"] == patient]
            if len(d):
                d.insert(0, "file", os.path.basename(f))
                rows.append(d)
        return _table(pd.concat(rows)) if rows else f"{patient} not in the pooled results of {out}"
    path = os.path.join(out, "lstm_ae_summary.csv")
    if not os.path.exists(path):
        return f"no lstm_ae_summary.csv in {out}"
    s = pd.read_csv(path)
    s = s[s["patient"] == patient]
    cols = [c for c in ["method", "status", "usable_seizures", "n_folds", "cv_auc_mean", "cv_detected", "cv_fpr_per_h",
                        "cv_p_vs_random", "holdout_auc", "holdout_detected", "holdout_fpr_per_h",
                        "holdout_p_vs_random"] if c in s]
    return _table(s[cols]) if len(s) else f"{patient} not in {path}"


def tool_compare_sops(mode="per-patient"):
    old = args.mode
    args.mode = mode
    try:
        return _table(_quiet(compare_sops, args))
    finally:
        args.mode = old


def tool_eeg_vs_position(part="cv", mode="per-patient"):
    old = args.mode
    args.mode = mode
    try:
        r = _quiet(eeg_vs_position, args, part)
    finally:
        args.mode = old
    return ("gain = AUC(logreg_posclock) - AUC(posclock) per patient; p_gain = one-sided Wilcoxon\n" + _table(r)
            if len(r) else "no results with both posclock and logreg_posclock")


def tool_list_recordings(patient):
    files = sorted(glob.glob(os.path.join(args.root, f"sub-{patient}", "eeg", "*_eeg.edf")))
    if not files:
        return f"no EDF files for {patient} under {args.root}"
    lines = []
    for f in files:
        ev = f.replace("_eeg.edf", "_events.tsv")
        n = 0
        if os.path.exists(ev):
            e = pd.read_csv(ev, sep="\t", encoding="utf-8-sig")
            if "trial_type" in e:
                n = int(e["trial_type"].astype(str).str.contains("seizure", case=False).sum())
        lines.append(f"{f}  seizures: {n}")
    return "\n".join(lines)


def tool_predict(edf_path, patient, sop=None, method="gat", mode="per-patient"):
    sop = int(sop or args.sop_list[0])
    old = args.mode
    args.mode = mode
    use_sop(args, sop)
    try:
        log = io.StringIO()
        with contextlib.redirect_stdout(log):
            scores, alarms = predict_edf(edf_path, patient, method=method)
        plt.close("all")
    finally:
        args.mode = old
        use_sop(args, sop)
    table = alarms.head(30).to_string(index=False) if hasattr(alarms, "to_string") else str(alarms)
    return f"{len(alarms)} alarms\n{table}\n--- predict_edf output ---\n" + log.getvalue()[-3000:]


AGENT_TOOLS = {
    "list_runs": (tool_list_runs, "List the result folders that exist (one per mode and SOP).", {}),
    "overall_results": (tool_overall, "Overall results of every method for one SOP and mode: seizures predicted, "
                        "sensitivity, false alarms per hour, AUC, p vs random predictor.",
                        {"sop": ("integer", "SOP in minutes, e.g. 10, 20 or 30"),
                         "mode": ("string", "per-patient or pooled")}),
    "patient_results": (tool_patient, "Results of every method for one patient, one SOP and mode.",
                        {"patient": ("string", "patient id, e.g. chb01"), "sop": ("integer", "SOP in minutes"),
                         "mode": ("string", "per-patient or pooled")}),
    "compare_sops": (tool_compare_sops, "All SOPs side by side for every method.",
                     {"mode": ("string", "per-patient or pooled")}),
    "eeg_vs_position": (tool_eeg_vs_position, "Test whether the EEG adds over position + clock time "
                        "(logreg_posclock vs posclock), per SOP.",
                        {"part": ("string", "cv or hidden"), "mode": ("string", "per-patient or pooled")}),
    "list_recordings": (tool_list_recordings, "List a patient's EDF recordings and how many seizures each has.",
                        {"patient": ("string", "patient id, e.g. chb01")}),
    "predict_recording": (tool_predict, "Score one EDF recording with a saved GAT or LSTM autoencoder and "
                          "return the alarm times. Slow (reads the whole file).",
                          {"edf_path": ("string", "full path of the .edf file"), "patient": ("string", "patient id"),
                           "sop": ("integer", "SOP in minutes of the saved model"),
                           "method": ("string", "gat or lstm"), "mode": ("string", "per-patient or pooled")}),
}
AGENT_REQUIRED = {"patient_results": ["patient"],
                  "list_recordings": ["patient"], "predict_recording": ["edf_path", "patient"]}


def _tool_schemas():
    out = []
    for name, (_, desc, params) in AGENT_TOOLS.items():
        props = {p: {"type": t, "description": d} for p, (t, d) in params.items()}
        out.append({"type": "function", "function": {
            "name": name, "description": desc,
            "parameters": {"type": "object", "properties": props, "required": AGENT_REQUIRED.get(name, [])}}})
    return out


AGENT_SYSTEM = f"""You help a student with a seizure PREDICTION project on the CHB-MIT EEG dataset.
Setup: 18 bipolar channels, 5 s windows, SWT wavelet features + channel correlations.
Preictal = [onset - SOP - 5 min, onset - 5 min) (SPH 5 min), SOP in {args.sop_list} min; interictal >= 4 h from seizures.
Methods: logreg, gbm (supervised); gat (GAT-LSTM autoencoder), lstm (LSTM autoencoder), trained on interictal only;
controls without EEG: posclock (position in recording + clock time), position, timeofday; logreg_posclock = EEG + posclock.
Alarms: threshold at <= {args.max_fpr} false alarms per hour; compared with a random predictor (p < 0.05 = better than chance).
A method only shows a real EEG signal if it beats the controls without EEG; AUC below 0.5 means the ranking is inverted.
Use the tools to get numbers; never invent numbers. Say which SOP, mode and part (cv or hidden) each number comes from.
Answer briefly and in plain language."""


def ask_agent(question, history=None, max_steps=8, verbose=True):
    """Ask the agent a question. Pass the returned history back in to keep the conversation going."""
    messages = history or [{"role": "system", "content": AGENT_SYSTEM}]
    messages.append({"role": "user", "content": question})
    for _ in range(max_steps):
        r = nvidia_client.chat.completions.create(model=AGENT_MODEL, messages=messages, tools=_tool_schemas(),
                                                  tool_choice="auto", temperature=0.2, max_tokens=1500)
        msg = r.choices[0].message
        calls = msg.tool_calls or []
        messages.append({"role": "assistant", "content": msg.content or "",
                         **({"tool_calls": [c.model_dump() for c in calls]} if calls else {})})
        if not calls:
            print(msg.content)
            return messages
        for c in calls:
            name = c.function.name
            try:
                kw = json.loads(c.function.arguments or "{}")
                result = AGENT_TOOLS[name][0](**kw) if name in AGENT_TOOLS else f"unknown tool {name}"
            except Exception as e:                       # give the error back to the model instead of stopping
                result = f"error: {type(e).__name__}: {e}"
            if verbose:
                print(f"[tool] {name}({c.function.arguments})")
            messages.append({"role": "tool", "tool_call_id": c.id, "content": str(result)[:8000]})
    print("stopped after", max_steps, "tool rounds")
    return messages


In [ ]:
chat = ask_agent("Which method works best for SOP 30 in per-patient mode, and does the EEG beat the position + time control?")
# chat = ask_agent("And for chb01 only?", chat)
# chat = ask_agent(r"Run the GAT model of chb01 (SOP 10) on C:\dataset chb\.nemar\sub-chb01\eeg\sub-chb01_task-rest_run-26_eeg.edf")
